In [ ]:
import __main__
import sys, os
project_root = os.path.abspath("..")  # adjust if notebook is elsewhere
sys.path.insert(0, project_root)
from typing import Dict, List, Literal, Tuple, Optional, Any
import logging
import math
import gc
import pickle
import time
import json
import random
import yaml
from datetime import datetime
from pathlib import Path

import category_encoders as ce
from matplotlib import pyplot as plt
from momentfm import MOMENTPipeline
import numexpr as ne # makes numpy operations faster
import numpy as np
import optuna
import pandas as pd
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.random_projection import GaussianRandomProjection

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    # print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

from benchmarks.ts2vec_runner import run_ts2vec, log_ts2vec_results
from benchmarks.timevae_runner import run_timevae, log_timevae_results
from benchmarks.moment_runner import MomentRunner
from benchmarks.barlow_cnn_runner import BarlowCNNRunner

from methods.forecasting_module import TimeGPTForecaster, SARIMAXForecaster
from methods.cellsup import Cellsup, DeepClusterAndSwav
from methods.mlp_heads import _get_orthogonality_penalty, make_MLP_regression_head, evaluate_MLP_regressor, train_sup_head_per_encoder, train_sup_heads_joint

from scripts.timevae_script import run_timevae_block
from scripts.ts2vec_script import run_ts2vec_block
from scripts.moment_script import run_moment_block
from scripts.barlow_cnn_script import run_barlow_cnn_block
from scripts.config_loader import load_project_configuration, load_specific_method_params
from scripts.pipeline import load_the_data, split_data_to_labeled_unlabeled
from src.scripts.direct_preds_script import eval_mean_row, eval_custom_row, eval_random_row, eval_flattened

from utils.io_utils import JSONLogger, Notifiers, read_yaml_params, set_all_rand_seeds
from utils.metrics_utils import AutocorrMetrics, Preds, Losses, DimensionalityEstimator, ForecastUtils, SemiSupLearning
from utils.data_utils import Slicing, Bootstrapping, assign_encoder_weights, convert_numpy, select_top_X_features, Augmentations
from utils.model_utils import Decoder, ProjectionHead, TorchWrapper, schedule_learning_rate, norm_temp_xentropy_loss, profile_epoch

import encoders.autoencoders as ae
from encoders.cnn import CnnAutoencoder
from encoders.decentralized_encoders import HorizontalFedEncoder, VerticalFedEncoder
from encoders.latents import Latents
from encoders.lstm_network import LSTMModel, LSTMTrainer, Seq2SeqLSTM
import encoders.train_autoencoders as train_ae
from encoders.ts2vec_encoder import TS2VecEncoder

import param_config.config_paths as P

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


In [ ]:
"[RUN ME] Setup step"
best_params = yaml.safe_load(open(P.best_params_path))

cfg         = load_project_configuration(P.params_path, P.data_params_yaml_path, P.messager_yaml_path)
params      = cfg.params
data_params = cfg.data_params

set_all_rand_seeds(cfg.rand_seed)

X_train, X_test, y_train_scaled, y_test_scaled, window_size = load_the_data(
    cfg.desired_dataset,
    cfg.num_pages_to_use,
    cfg.do_we_scale_y,
    cfg.dataset_window,
    cfg.rand_seed,
    cfg.num_rows_per_page,
    cfg.params,
    cfg.label_frac,
    use_cache=False)
X_L, y_L, X_U, y_U, y_train_scaled, y_test_scaled, timevae_file_path = split_data_to_labeled_unlabeled(
    cfg.desired_dataset,
    P.interim_data_loc,
    cfg.data_splitting,
    cfg.label_frac,
    X_train, y_train_scaled,
    X_test, y_test_scaled,
    cfg.params,
    rand_seed=cfg.rand_seed)


In [ ]:
"[RUN SCRIPTS]"

if params["run_console"]["timevae"]:
    timevae_raw_params = load_specific_method_params(dataset_name=cfg.desired_dataset, method="timevae",
                                              best_params_dict=best_params, dataset_params=data_params)
    timevae_cfg = {"timevae": timevae_raw_params}
    timevae_losses, timevae_recon_loss_test, r2, metrics, model_cfg, train_cfg = run_timevae_block(
        X_train, X_test, y_train_scaled, y_test_scaled,
        timevae_file_path,
        params=timevae_cfg,
        desired_dataset=cfg.desired_dataset,
        window_size=window_size,
        device=device,
        force_train=False)

if params["run_console"]["ts2vec"]:
    ts2vec_raw_params = load_specific_method_params(dataset_name=cfg.desired_dataset, method="ts2vec",
                                             best_params_dict=best_params, dataset_params=data_params)
    ts2vec_cfg = {"ts2vec": ts2vec_raw_params}
    ts2vec_losses, r2, metrics, model_cfg, train_cfg = run_ts2vec_block(
        X_train, X_test, y_train_scaled, y_test_scaled,
        ts2vec_cfg,
        cfg.desired_dataset,
        window_size,
        device)

if params["run_console"]["moment"]:
    moment_raw_params = load_specific_method_params(dataset_name=cfg.desired_dataset, method="moment",
                                             best_params_dict=best_params, dataset_params=data_params)
    moment_cfg = {"moment": moment_raw_params}
    moment_losses, r2, metrics, model_cfg, train_cfg = run_moment_block(
        X_train, X_test, y_train_scaled, y_test_scaled,
        moment_cfg,
        cfg.desired_dataset,
        device)

if params["run_console"]["barlow_cnn"]:
    barlow_raw_params = load_specific_method_params(dataset_name=cfg.desired_dataset, method="barlow_cnn",
                                             best_params_dict=best_params, dataset_params=data_params)
    barlow_cfg = {"barlow_cnn": barlow_raw_params}
    barlow_cnn_losses, r2, metrics, barlow_recon_train, barlow_recon_test, model_cfg, train_cfg = run_barlow_cnn_block(
        X_train, X_test, y_train_scaled, y_test_scaled,
        barlow_cfg,
        desired_dataset=cfg.desired_dataset,
        device=device)


if params["run_console"]["direct_preds"]["mean_X"]:
    mean_losses, rf_model_mean = eval_mean_row(X_L, X_test, y_L, y_test_scaled, cfg, device)
if params["run_console"]["direct_preds"]["flatten_X"]:
    flat_losses, rf_model_flat = eval_flattened(X_L, X_test, y_L, y_test_scaled, cfg, device)
if params["run_console"]["direct_preds"]["custom_row"]:
    custom_row_number = -1
    custom_losses, rf_model_custom = eval_custom_row(X_L, X_test, y_L, y_test_scaled, cfg, device, custom_row_number)
if params["run_console"]["direct_preds"]["random_row"]:
    rand_losses, rf_model_rand = eval_random_row(X_L, X_test, y_L, y_test_scaled, cfg, device)


In [ ]:
{'dataset': ['beijing'],
'window': [256],
'moment': [0.323, ],
'barlow': [0.246],
'ts2vec': [0.776, ],
}

In [ ]:
"optuna hyperparameter search"

class OptunaTuner:
    "Hyperparameter tuner for various methods using Optuna"

    @staticmethod
    def timevae_optuna_objective(trial, X_train, X_test, y_train, y_test,
                        device, timevae_file_path, desired_dataset, log_file):
        latent_dim        = trial.suggest_categorical("latent_dim", [4, 8, 12, 16, 32])
        hidden_layers     = trial.suggest_categorical("hidden_layers", [[12,8,12], [32,16,8], [64,32,16], [64,64,64]])
        reconstruction_wt = trial.suggest_float("reconstruction_wt", 0.5, 3.5)
        batch_size        = trial.suggest_categorical("batch_size", [128])
        lr                = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
        train_epochs      = 100

        model_cfg = {
            "latent_dim": latent_dim,
            "hidden_layers": hidden_layers,
            "reconstruction_wt": reconstruction_wt}
        train_cfg = {
            "train_epochs": train_epochs,
            "lr": lr,
            "batch_size": batch_size}
        try:
            losses, r2, _, _, recon_loss_test, _, _ = run_timevae(
                X_train, X_test, y_train, y_test,
                timevae_file_path=timevae_file_path,
                device=device,
                batch_size=train_cfg["batch_size"],
                train_epochs=train_cfg["train_epochs"],
                lr_training=train_cfg["lr"],
                latent_dim=model_cfg["latent_dim"],
                hidden_layer_sizes=model_cfg["hidden_layers"],
                reconstruction_wt=model_cfg["reconstruction_wt"],
                desired_dataset=desired_dataset,
                force_train=True,)

            record = convert_numpy({
                **model_cfg,
                **train_cfg,
                "rmse": float(losses[0]),
                "r2": float(r2),
                "recon_loss": float(recon_loss_test)})
            Path(log_file).parent.mkdir(parents=True, exist_ok=True)
            with open(log_file, "a") as f:
                f.write(json.dumps(record) + "\n")

            return float(losses[0])

        except Exception as e:
            print("Trial failed:", e)
            return float("nan")

    @staticmethod
    def ts2vec_optuna_objective(trial):
        # Define hyperparameter search space
        hidden_dims = trial.suggest_categorical("hidden_dims", [12, 16, 20, 24])
        latent_dims = trial.suggest_categorical("latent_dims", [8, 12, 16, 20])
        depth       = trial.suggest_categorical("depth", [3, 4, 5])
        lr          = trial.suggest_categorical("lr", [0.001, 0.005])
        z_pooling   = trial.suggest_categorical("z_pooling_method", ["None"])#, "mean", "max"])
        batch_size  = 1024
        epochs      = 100

        model_cfg = {
            "hidden_dims": hidden_dims,
            "latent_dims": latent_dims,
            "depth": depth,
            "z_pooling_method": z_pooling}

        train_cfg = {
            "lr": lr,
            "batch_size": batch_size,
            "epochs": epochs,
            "patience": 25,
            "window_size": window_size} # use your predefined window_size
        try:
            losses, r2, metrics = run_ts2vec(X_train, X_test, y_train_scaled, y_test_scaled,
                                            model_cfg=model_cfg, train_cfg=train_cfg, device=device)
            # Log results
            record = {**model_cfg, **train_cfg, "test_rmse": losses[0], "r2": r2}
            with open(P.ts2vec_hyperparam_file, "a") as f:
                f.write(json.dumps(record) + "\n")
            return losses[0]  # Optimize RMSE
        except Exception as e:
            print("Trial failed:", e)
            return float("inf")

    @staticmethod
    def barlow_optuna_objective(trial):
        # Define hyperparameter search space
        latent_dim   = trial.suggest_categorical("latent_dim", [8, 12, 16, 32])
        channels1    = trial.suggest_categorical("channels1", [8, 12, 16, 32])
        channels2    = trial.suggest_categorical("channels2", [8, 16, 32])
        kernel_size  = trial.suggest_categorical("kernel_size", [3, 5, 7, 9])
        pool_kernel  = trial.suggest_categorical("pool_kernel", [2, 3, 4])
        ssl_lambda   = trial.suggest_categorical("ssl_lambda", [0.01, 0.1, 0.5, 1.0])
        ssl_weight   = trial.suggest_categorical("ssl_weight", [0.1, 0.5, 1.0])
        recon_weight = trial.suggest_categorical("recon_weight", [0.1, 0.5, 1.0])
        augment_const = trial.suggest_categorical("augment_const", [0.05, 0.1, 0.2])
        lr            = trial.suggest_categorical("lr", [0.001, 0.005, 0.01])
        head_dims_list= trial.suggest_categorical("head_dims_list", [[64, 32], [128, 64]])
        epochs       = 100
        rand_seed    = 42

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        model_cfg = {
            "latent_dim": latent_dim,
            "channels": [channels1, channels2],
            "kernel_size": kernel_size,
            "pool_kernel": pool_kernel,
            "ssl_lambda": ssl_lambda,
            "ssl_weight": ssl_weight,
            "recon_weight": recon_weight,
            "augment_const": augment_const,
            "head_dims_list": head_dims_list}

        train_cfg = {"epochs": epochs, "lr": lr, "rand_seed": rand_seed}

        try:
            losses, r2, metrics, recon_train, recon_test = BarlowCNNRunner.run_barlow_cnn(
                X_train, X_test, y_L, y_test_scaled,
                model_cfg=model_cfg, train_cfg=train_cfg, device=device)

            # Log trial
            record = {**model_cfg, **train_cfg,
                    "test_rmse": losses[0], "r2": r2,
                    "final_recon_train": recon_train, "final_recon_test": recon_test}
            with open(P.barlow_hyperparam_file, "a") as f:
                f.write(json.dumps(record) + "\n")
            return losses[0]  # Optimize RMSE
        except Exception as e:
            print(f"[Trial failed] {e}")
            return float("inf")

    @staticmethod
    def moment_optuna_objective(trial, repeats=1):
        hidden_layers   = trial.suggest_categorical("hidden_layers", [[64, 32, 8], [128,64,16], [256,128,64], [512,256,64,16]])
        unfreeze_last_n = trial.suggest_int("unfreeze_last_n", 1, 2)
        dropout    = trial.suggest_categorical("dropout", [0.0, 0.1, 0.2])
        lr_encoder = trial.suggest_categorical("lr_encoder", [1e-4, 5e-4, 1e-3])
        lr_head    = trial.suggest_categorical("lr_head", [1e-3, 5e-3, 1e-2])
        batch_size = 256
        epochs     = trial.suggest_categorical("epochs", [5, 10, 20]) #10

        losses_accum = 0
        r2_accum     = 0

        for _ in range(repeats):
            # Set random seeds per repeat
            # rng       = np.random.default_rng()
            # rand_seed = rng.integers(0, 2**32 - 1)
            # torch.manual_seed(rand_seed)
            # np.random.seed(rand_seed)
            # random.seed(rand_seed)
            # if device.type.startswith("cuda"):
            #     torch.cuda.manual_seed(rand_seed)
            #     torch.cuda.manual_seed_all(rand_seed)

            # Fresh head per repeat
            head = make_MLP_regression_head(latent_dim, hidden_layers, y_train_scaled, dropout, device)

            model_cfg = {
                'latent_dim': latent_dim,
                'hidden_layers': hidden_layers,
                'dropout': dropout,
                'unfreeze_last_n': unfreeze_last_n,
                'model_name': params['moment']['model_name'],
                'task_name': params['moment']['model_task']}
            train_cfg = {
                'epochs': epochs,
                'batch_size': batch_size,
                'lr_encoder': lr_encoder,
                'lr_head': lr_head,
                'fine_tune': True}

            # Unfreeze last N blocks
            MomentRunner._unfreeze_last_n_blocks(moment_model, unfreeze_last_n)

            # Train and evaluate
            losses, r2, metrics = MomentRunner.run_moment(
                X_train, X_test, y_train_scaled, y_test_scaled,
                model_cfg=model_cfg,
                train_cfg=train_cfg,
                device=device,
                moment_model=moment_model,
                head=head)

            losses_accum += losses[3]  # NN test RMSE
            r2_accum += r2

        # Average over repeats
        losses_avg = [*losses[:3], losses_accum / repeats]
        r2_avg     = r2_accum / repeats

        # Log trial
        record = {**model_cfg, **train_cfg, "test_rmse": losses_avg[3], "r2": r2_avg}
        with open(P.moment_hyperparam_file, "a") as f:
            f.write(json.dumps(record) + "\n")
        return losses_avg[3]  # optimize NN test RMSE


if params["run_console"]["ts2vec"]:
    study = optuna.create_study(direction="minimize")
    study.optimize(OptunaTuner.ts2vec_optuna_objective, n_trials=20)

    print("Best trial:")
    print(study.best_trial.params)
    print("Best RMSE:", study.best_value)

if params["run_console"]["timevae"]:
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    study  = optuna.create_study(direction="minimize", pruner=pruner, study_name="TimeVAE_HPO")
    study.optimize(
        lambda trial: OptunaTuner.timevae_optuna_objective(
            trial,
            X_train, X_test,
            y_train_scaled, y_test_scaled,
            device=device,
            timevae_file_path=timevae_file_path,
            desired_dataset=cfg.desired_dataset,
            log_file=P.timevae_hyperparam_file),
        n_trials=20,
        n_jobs=1)   # GPU → keep at 1 unless you have multiple GPUs
    print("Best hyperparameters:", study.best_params)
    print("Best RMSE:", study.best_value)

if params["run_console"]["barlow_cnn"]:
    study = optuna.create_study(direction="minimize")
    study.optimize(OptunaTuner.barlow_optuna_objective, n_trials=20)
    print("Best trial params:", study.best_trial.params)
    print("Best RMSE:", study.best_value)

if params["run_console"]["moment"]:
    moment_model = MOMENTPipeline.from_pretrained(
        f"AutonLab/{params['moment']['model_name']}",
        model_kwargs={'task_name': params['moment']['model_task'], 'n_channels': X_train.shape[2]}).to(device)
    moment_model.eval()  # start frozen

    patch_size = getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None)
    X_sample   = MomentRunner.pad_to_moment_patch_size(torch.tensor(X_train[:1], dtype=torch.float32), patch_size).permute(0, 2, 1).to(device)
    latent_dim = moment_model.embed(x_enc=X_sample).embeddings.shape[1]

    # Freeze all parameters
    for p in moment_model.parameters():
        p.requires_grad = False

    # Create study and optimize
    study = optuna.create_study(direction="minimize")
    study.optimize(OptunaTuner.moment_optuna_objective, n_trials=20)

    print("Best MOMENT config:", study.best_trial.params)
    print("Best test RMSE:", study.best_value)


In [ ]:
"fed ts2vec block"

class BaseEncoder:
    """Federated encoder interface."""
    def fit(self, X: np.ndarray, y_train=None, y_test=None) -> Any:
        raise NotImplementedError

    def encode(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError

class TS2VecFed(BaseEncoder):
    def __init__(self, params: dict, device: str):
        self.p = params
        self.device = device
        self.enc = TS2VecEncoder(
            z_pooling=self.p.get("z_pooling_method", "mean"),
            lr=self.p.get("lr_encoder", 1e-3),
            device=self.device,
            patience=self.p.get("patience", 20))

    def fit(self, X: np.ndarray, y_train=None, y_test=None):
        if isinstance(X, torch.Tensor):
            X = X.cpu().numpy()
        X = X.astype(np.float32)
        # 1. Run the TS2Vec training function
        self.enc.fit_ts2vec( # Changed: Function is called without using return statement
            X,
            hidden_dims=self.p["hidden_dims"],
            output_dims=self.p["latent_dims"],
            depth=self.p["depth"],
            batch_size=self.p["batch_size"],
            n_epochs=self.p["epochs"])
        # 2. Return None explicitly to satisfy the federated framework interface
        return None # <--- FIX APPLIED HERE

    def encode(self, X: np.ndarray) -> np.ndarray:
        if isinstance(X, torch.Tensor):
            X = X.cpu().numpy()
        X = X.astype(np.float32)
        return self.enc.encode(X)

# ---------------- Federated Setup ----------------
best_params    = yaml.safe_load(open(P.best_params_path))
encoder_choice = params["federated"]["which_encoder"].lower()
p              = best_params[cfg.desired_dataset][encoder_choice]

type_of_split  = "vertical" #params["federated"].get("type_of_split", "vertical")
NUM_FED_SPLITS = 8 #params["federated"].get("num_splits", 1)

if type_of_split == "horizontal":
    fed_object = HorizontalFedEncoder(num_splits=NUM_FED_SPLITS)
else:
    fed_object = VerticalFedEncoder(num_splits=NUM_FED_SPLITS)

# Select encoder class
EncoderClass = {"ts2vec": TS2VecFed}
                #, "moment": MomentFed, "barlow_cnn": BarlowCNNFed}.get(encoder_choice)
if EncoderClass is None:
    raise ValueError(f"Unknown encoder: {encoder_choice}")

# ---------------- Builder / Fit Functions ----------------
def encoder_builder():
    """Return a new encoder instance per split."""
    if encoder_choice == "ts2vec":
        return TS2VecFed(p, device)
    return EncoderClass(p, device)

def fit_function(enc, Xsplit):
    """Fit encoder on its split. Handles torch -> numpy internally."""
    enc.fit(Xsplit)  # y_train/y_test deferred for TimeVAE

# ---------------- Federated Encoding ----------------
Z_train_list, Z_test_list = fed_object.encode_federated(
    X_train, X_test,
    y_train_scaled, y_test_scaled,
    encoder_builder=encoder_builder,
    fit_function=fit_function,
    batch_size=p["batch_size"])

Z_train_cat, Z_test_cat = fed_object.combine_latents(Z_train_list, Z_test_list)

# ---------------- Evaluation ----------------
ts2vec_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train_cat, y_train_scaled, Z_test_cat, y_test_scaled)

print(ts2vec_fed_losses)
print(f"R²: {rf_model.score(Z_test_cat, y_test_scaled):.3f}")


In [ ]:
"timevae fed block"
from timevae_torch.src.vae.timevae import TimeVAE
from timevae_torch.src.vae.vae_utils import train_vae, get_posterior_samples

type_of_split = "horizontal"  # "vertical" or "horizontal"
NUM_FED_SPLITS= 3           # number of splits for either FL type

timevae_raw_params = load_specific_method_params(dataset_name=cfg.desired_dataset, method="timevae",
                                                 best_params_dict=best_params, dataset_params=data_params)
seq_len       = X_train.shape[1]
full_feat_dim = X_train.shape[2]  # total features
num_samples   = X_train.shape[0]

if type_of_split == "vertical":
    feat_per_split = full_feat_dim // NUM_FED_SPLITS
    splits = []
    start  = 0
    for _ in range(NUM_FED_SPLITS):
        end   = start + feat_per_split
        splits.append((start, end))
        start = end
    splits[-1]     = (splits[-1][0], full_feat_dim)
    X_train_splits = [X_train[:, :, s:e] for (s, e) in splits]
    X_test_splits  = [X_test[:, :, s:e]  for (s, e) in splits]

elif type_of_split == "horizontal":
    train_splits   = np.array_split(np.arange(X_train.shape[0]), NUM_FED_SPLITS)
    test_splits    = np.array_split(np.arange(X_test.shape[0]), NUM_FED_SPLITS)
    X_train_splits = [X_train[s, :, :] for s in train_splits]
    X_test_splits  = [X_test[s, :, :]  for s in test_splits]

class TimeVAEFed:
    def __init__(self, params: dict, device: str):
        self.p      = params
        self.device = torch.device(device)
        self.model  = TimeVAE(
            seq_len   =self.p["seq_len"],
            feat_dim  =self.p["feat_dim"],
            latent_dim=self.p["latent_dim"],
            hidden_layer_sizes=self.p["hidden_layers"],
            reconstruction_wt =self.p["reconstruction_wt"],
            batch_size=self.p["batch_size"]).to(self.device)

    def fit(self, X: np.ndarray):
        X_np = np.array(X, dtype=np.float32)
        train_vae(
            vae=self.model,
            train_data=X_np,
            max_epochs=self.p["epochs"],
            lr=self.p["lr_training"],
            verbose=0)

    def encode(self, X: np.ndarray) -> np.ndarray:
        X_np = np.array(X, dtype=np.float32)
        z    = get_posterior_samples(self.model, X_np)
        return z

latent_train_list = []
latent_test_list  = []

for i, (Xtr, Xte) in enumerate(zip(X_train_splits, X_test_splits)):
    if type_of_split == "vertical":
        split_feat_dim = Xtr.shape[2]
        base_h         = timevae_raw_params["hidden_layers"]
        hidden_layers  = [max(2, int(h * split_feat_dim / full_feat_dim)) for h in base_h]
        feat_dim       = split_feat_dim
    else:  # horizontal
        hidden_layers = timevae_raw_params["hidden_layers"]
        feat_dim      = full_feat_dim

    params_split = {
        "seq_len": seq_len,
        "feat_dim": feat_dim,
        "latent_dim": timevae_raw_params["latent_dim"],
        "hidden_layers": hidden_layers,
        "reconstruction_wt": timevae_raw_params["reconstr_wt"],
        "batch_size": timevae_raw_params["batch_size"],
        "lr_training": timevae_raw_params["lr"],
        "epochs": timevae_raw_params["epochs"],}
    print(f"[Split {i}] feat_dim={feat_dim}, hidden_layers={hidden_layers}")

    enc = TimeVAEFed(params=params_split, device=device)
    enc.fit(Xtr)
    ztr = enc.encode(Xtr)
    zte = enc.encode(Xte)
    latent_train_list.append(ztr)
    latent_test_list.append(zte)

    del enc, ztr, zte
    if device.type == "cuda":
        torch.cuda.empty_cache()

if type_of_split == "vertical":
    Z_train_cat = np.concatenate(latent_train_list, axis=1)  # features
    Z_test_cat  = np.concatenate(latent_test_list, axis=1)
else:  # horizontal
    Z_train_cat = np.concatenate(latent_train_list, axis=0)  # samples
    Z_test_cat  = np.concatenate(latent_test_list, axis=0)

print(f"Final shape: z (train)={Z_train_cat.shape}, z (test)={Z_test_cat.shape}")
# ---------------- Evaluation ----------------
timevae_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train_cat, y_train_scaled, Z_test_cat, y_test_scaled)
print(timevae_fed_losses)

_, seq_len, latent_dim = Z_train_cat.shape
n_test      = Z_test_cat.shape[0]
Z_test_flat = Z_test_cat.reshape(n_test, seq_len * latent_dim)
print(f"R²: {rf_model.score(Z_test_flat, y_test_scaled):.3f}")


In [ ]:
"moment fed"

def run_moment_fed_block(X_train, X_test, y_train_scaled, y_test_scaled, params: dict,
                         type_of_split="vertical", num_splits=3, device="cpu"):
    """Run MOMENT in federated mode: encode splits independently, combine latents, then regress."""
    # ----- SPLIT DATA -----
    _, _, full_feat_dim = X_train.shape
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

    if type_of_split == "vertical":
        feat_per_split = full_feat_dim // num_splits
        splits = []
        start = 0
        for _ in range(num_splits):
            end = start + feat_per_split
            splits.append((start, end))
            start = end
        splits[-1] = (splits[-1][0], full_feat_dim)
        X_train_splits = [X_train_tensor[:, :, s:e] for (s, e) in splits]
        X_test_splits  = [X_test_tensor[:, :, s:e]  for (s, e) in splits]
    else:
        train_splits = np.array_split(np.arange(X_train_tensor.shape[0]), num_splits)
        test_splits  = np.array_split(np.arange(X_test_tensor.shape[0]), num_splits)

        X_train_splits = [X_train_tensor[s, :, :] for s in train_splits]
        X_test_splits  = [X_test_tensor[s, :, :] for s in test_splits]

    # ----- PAD SPLITS FOR MOMENT PATCH SIZE -----
    temp_model = MOMENTPipeline.from_pretrained(f"AutonLab/{params['moment']['model_name']}")
    patch_size = getattr(temp_model.tokenizer, "patch_size", None) or getattr(temp_model.tokenizer, "patch_len", None)
    del temp_model

    def pad_and_permute(split):
        padded = MomentRunner.pad_to_moment_patch_size(split, patch_size)
        return padded.permute(0, 2, 1)  # B, C, T

    X_train_splits = [pad_and_permute(s) for s in X_train_splits]
    X_test_splits  = [pad_and_permute(s) for s in X_test_splits]

    # ----- ENCODE SPLITS -----
    latent_train_list = []
    latent_test_list  = []

    for i, (Xtr, Xte) in enumerate(zip(X_train_splits, X_test_splits)):
        print(f"[Split {i}] Xtr shape: {Xtr.shape}")
        encoder = MOMENTPipeline.from_pretrained(
            f"AutonLab/{params['moment']['model_name']}",
            model_kwargs={'task_name':'regression', 'n_channels': Xtr.shape[1]}).to(device)
        encoder.eval()

        with torch.no_grad():
            ztr = MomentRunner.encode_x_to_z_in_batches(encoder, Xtr.to(device))
            zte = MomentRunner.encode_x_to_z_in_batches(encoder, Xte.to(device))

        latent_train_list.append(ztr.cpu())
        latent_test_list.append(zte.cpu())
        del encoder
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # ----- COMBINE LATENTS -----
    if type_of_split == "vertical":
        Z_train = torch.cat(latent_train_list, dim=1).numpy()  # features
        Z_test  = torch.cat(latent_test_list,  dim=1).numpy()
    else:
        Z_train = torch.cat(latent_train_list, dim=0).numpy()  # samples
        Z_test  = torch.cat(latent_test_list,  dim=0).numpy()

    print(f"Final Z shape: train={Z_train.shape}, test={Z_test.shape}")

    # ----- DOWNSTREAM REGRESSION -----
    moment_fed_losses, rf_model = Preds().evaluate_models_on_dataset(
        Z_train, y_train_scaled, Z_test, y_test_scaled)
    print(f"Moment FED losses: {[f'{x:.3f}' for x in moment_fed_losses]}")
    print(f"R²: {rf_model.score(Z_test.reshape(Z_test.shape[0], -1), y_test_scaled):.3f}")
    return moment_fed_losses, rf_model, Z_train, Z_test

moment_fed_losses, rf_model, Z_train, Z_test = run_moment_fed_block(
    X_train, X_test, y_train_scaled, y_test_scaled,
    params=params,
    type_of_split="horizontal",
    num_splits=3,
    device=device)


In [ ]:
params["barlow_cnn"]

In [ ]:
# ============================================
# FLEXIBLE Barlow CNN FED BLOCK + LATENT COMBINE
# ============================================
def build_barlow_configs(barlow_dict: dict):
    """Split flexible Barlow dict -> (model_cfg, train_cfg), no hardcoding."""
    model_keys = {
        "latent_dim", "channels", "channels_1", "channels_2",
        "kernel_size", "pool_kernel",
        "ssl_lambda", "ssl_weight",
        "recon_weight", "augment_const",
        "head_dims_list"
    }

    train_keys = {"epochs", "lr", "rand_seed"}

    model_cfg = {}
    train_cfg = {}

    for k, v in barlow_dict.items():
        key = k.lower()
        if key in {x.lower() for x in train_keys}:
            train_cfg[key] = v
        else:
            model_cfg[key] = v

    if "channels_1" in model_cfg and "channels_2" in model_cfg:
        model_cfg["channels"] = [model_cfg.pop("channels_1"),
                                 model_cfg.pop("channels_2")]

    return model_cfg, train_cfg


def run_barlow_fed_block(X_train, X_test, y_train_scaled, y_test_scaled,
                         params: dict, type_of_split="vertical",
                         num_splits=3, device=device):
    model_cfg, train_cfg = build_barlow_configs(params["barlow_cnn"])
    if "rand_seed" not in train_cfg:
        train_cfg["rand_seed"] = 42  # fallback

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

    num_samples, seq_len, full_feat_dim = X_train.shape

    # ------------- SPLITTING -------------
    if type_of_split == "vertical":
        feat_per_split = full_feat_dim // num_splits
        splits, start = [], 0
        for _ in range(num_splits):
            end = start + feat_per_split
            splits.append((start, end))
            start = end
        splits[-1] = (splits[-1][0], full_feat_dim)
        X_train_splits = [X_train_tensor[:, :, s:e] for (s, e) in splits]
        X_test_splits  = [X_test_tensor[:, :, s:e] for (s, e) in splits]
        y_train_splits = [y_train_scaled] * num_splits
        y_test_splits  = [y_test_scaled] * num_splits
    else:  # horizontal
        train_splits = np.array_split(np.arange(num_samples), num_splits)
        test_splits  = np.array_split(np.arange(X_test_tensor.shape[0]), num_splits)
        X_train_splits = [X_train_tensor[idx] for idx in train_splits]
        X_test_splits  = [X_test_tensor[idx] for idx in test_splits]
        y_train_splits = [y_train_scaled[idx] for idx in train_splits]
        y_test_splits  = [y_test_scaled[idx] for idx in test_splits]

    latent_train_list = []
    latent_test_list  = []

    # ------------- TRAIN + ENCODE EACH SPLIT -------------
    for i, (Xtr, Xte, ytr, yte) in enumerate(
        zip(X_train_splits, X_test_splits, y_train_splits, y_test_splits)):
        print(f"[Split {i}] Xtr = {tuple(Xtr.shape)}")

        # ---- train split CNN ----
        cnn = CnnAutoencoder(
            n_features=Xtr.shape[2],       # channels = split width
            n_timesteps=Xtr.shape[1],
            latent_dim=model_cfg["latent_dim"],
            channels=model_cfg["channels"],
            kernel_size=model_cfg["kernel_size"],
            pool_kernel=model_cfg["pool_kernel"]).to(device)

        BarlowCNNRunner.train_encoder(
            cnn, Xtr.to(device),
            epochs=train_cfg["epochs"],
            lr=train_cfg["lr"],
            ssl_lambda=model_cfg["ssl_lambda"],
            ssl_weight=model_cfg["ssl_weight"],
            recon_weight=model_cfg["recon_weight"],
            augment_const=model_cfg["augment_const"],
            device=device,
            rand_seed=train_cfg["rand_seed"])

        # ---- encode split ----
        cnn.eval()
        with torch.no_grad():
            ztr = cnn.encode(Xtr.to(device)).cpu()
            zte = cnn.encode(Xte.to(device)).cpu()

        latent_train_list.append(ztr)
        latent_test_list.append(zte)

    # ------------- COMBINE LATENTS -------------
    if type_of_split == "vertical":
        Z_train = torch.cat(latent_train_list, dim=1).numpy()
        Z_test  = torch.cat(latent_test_list, dim=1).numpy()
    else:
        Z_train = torch.cat(latent_train_list, dim=0).numpy()
        Z_test  = torch.cat(latent_test_list, dim=0).numpy()
    return Z_train, Z_test

Z_train, Z_test = run_barlow_fed_block(
    X_train, X_test,
    y_train_scaled, y_test_scaled,
    params=cfg.params,
    type_of_split="horizontal",  # or "vertical"
    num_splits=3,
    device=device)

print("Z_train:", Z_train.shape)
print("Z_test:", Z_test.shape)

    # ----- DOWNSTREAM REGRESSION -----
barlow_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train, y_train_scaled, Z_test, y_test_scaled)
print(f"Barlow FED losses: {[f'{x:.3f}' for x in barlow_fed_losses]}")
print(f"R²: {rf_model.score(Z_test.reshape(Z_test.shape[0], -1), y_test_scaled):.3f}")


In [ ]:
"[old?] federated section"
encoder_choice = params["federated"]["which_encoder"]
best_params    = yaml.safe_load(open(P.best_params_path))
p = best_params[cfg.desired_dataset][encoder_choice]

type_of_split  = "vertical" #params["federated"]["type_of_split"]
NUM_FED_SPLITS = 1 #params["federated"]["num_splits"]

if type_of_split == "horizontal":
    fed = HorizontalFedEncoder(num_splits=NUM_FED_SPLITS)
elif type_of_split == "vertical":
    fed = VerticalFedEncoder(num_splits=NUM_FED_SPLITS)
    ts2vec_latent_dims = p["latent_dims"] // NUM_FED_SPLITS
    ts2vec_hidden_dims = p["hidden_dims"] // NUM_FED_SPLITS
    ts2vec_depth       = p["depth"] // NUM_FED_SPLITS
else:
    raise ValueError("Invalid type_of_split")

def federated_encoder_builder():
    if encoder_choice.lower() == "ts2vec":
        return TS2VecEncoder(z_pooling=p["z_pooling_method"], lr=p["lr_encoder"], device=device, patience=p["patience"])
    elif encoder_choice.lower() == "timevae":
        pass
    elif encoder_choice.lower() == "moment":
        pass
    elif encoder_choice.lower() == "barlow_cnn":
        pass

def fit_function(encoder, X_split):
    if encoder_choice.lower() == "ts2vec":
        encoder.fit_ts2vec(X_split.cpu().numpy(), hidden_dims=p["ts2vec_hidden_dims"], output_dims=p["ts2vec_latent_dims"],
                           depth=p["ts2vec_depth"], batch_size=p["batch_size"], n_epochs=p["epochs"])

Z_train_list, Z_test_list = fed.encode_federated(X_train, X_test, y_train_scaled, y_test_scaled,
                            federated_encoder_builder, fit_function, batch_size=p["batch_size"])
Z_train_cat, Z_test_cat   = fed.combine_latents(Z_train_list, Z_test_list)

ts2vec_fed_losses, rf_model= Preds().evaluate_models_on_dataset(Z_train_cat, y_train_scaled, Z_test_cat,  y_test_scaled)
r2              = rf_model.score(Z_test_cat, y_test_scaled)
print(ts2vec_fed_losses)
print(f"R²: {r2:.3f}")


In [ ]:
"dimensionality"
# intrinsic_dim_est = DimensionalityEstimator.estimate_intrinsic_dim_mle(X_train)
# print("Levina-Bickel MLE Intrinsic dim estimate:", intrinsic_dim_est)

# latent_dim_est = DimensionalityEstimator.estimate_intrinsic_dim_skdim(X_train, method="mle", K=15)
# print(f"Skdim estim. intrinsic dim: {latent_dim_est:.2f}")

# avg_dimensionality = (intrinsic_dim_est + latent_dim_est) / 2
# latent_dim = int(np.ceil(avg_dimensionality * 1.3))
# print(f"Chosen latent dim (1.3x avg): {latent_dim}")

# pca_latent_dim = DimensionalityEstimator.count_active_latents(timevae_model, X_train.reshape(X_train.shape[0], -1), kl_threshold=0.99)
# print(f"PCA-based latent dim for 95% energy: {pca_latent_dim}")


In [ ]:
class ARDSeqVAE(nn.Module):
    """
    Sequence VAE with ARD-style latent dimension selection
    X shape: (batch, seq_len, features)
    """
    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder_rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # ARD parameters
        self.log_alpha = nn.Parameter(torch.zeros(latent_dim))

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn = nn.LSTM(hidden_dim, input_dim, batch_first=True)

    def encode(self, x):
        _, (h_n, _) = self.encoder_rnn(x)  # h_n: (1, batch, hidden_dim)
        h_n = h_n.squeeze(0)
        mu = self.fc_mu(h_n)
        logvar = self.fc_logvar(h_n)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, seq_len):
        h = F.relu(self.fc_dec(z)).unsqueeze(1).repeat(1, seq_len, 1)
        out, _ = self.decoder_rnn(h)
        return out

    def forward(self, x):
        seq_len = x.size(1)
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, seq_len)
        return x_hat, mu, logvar, z

def ard_seq_vae_loss(x, x_hat, mu, logvar, log_alpha):
    recon_loss = F.mse_loss(x_hat, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - log_alpha - (mu**2 + torch.exp(logvar)) / torch.exp(log_alpha))
    return recon_loss + kl


latent_dim = 40
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)

model = ARDSeqVAE(
    input_dim=X_train_tensor.shape[2],
    latent_dim=latent_dim).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(200):
    x_hat, mu, logvar, z = model(X_train_tensor)
    loss = ard_seq_vae_loss(X_train_tensor, x_hat, mu, logvar, model.log_alpha)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print(epoch, loss.item())

# inspect
log_alpha = model.log_alpha.detach().cpu()
print("active dims:", (log_alpha < 8).sum().item())


In [ ]:
"TimeVAE hyperparameter search"

if params["run_console"]["timevae"]:
    latent_dims_list        = [8, 12, 16, 32]
    hidden_layer_sizes_list = [[32,16,8], [64,32,16], [128,64,96], [64,64,64]]
    lr_list                 = [1e-4, 5e-4, 1e-3]
    batch_size_list         = [256]
    reconstruction_wt_list  = [0.5, 3.5]
    train_epochs            = 100

    repeats = 1 #2
    counter = 0

    for ld in latent_dims_list:
        for hls in hidden_layer_sizes_list:
            for lr in lr_list:
                for bs in batch_size_list:
                    for rw in reconstruction_wt_list:
                        model_cfg = {
                            "hidden_layers": hls,
                            "latent_dim": ld,
                            "reconstruction_wt": rw}
                        train_cfg = {
                            "train_epochs": train_epochs,
                            "lr": lr,
                            "batch_size": bs}

                        rmse_accum  = linreg_accum  = catboost_accum = 0
                        r2_accum    = runtime_accum = params_accum   = 0
                        flops_accum = mem_accum     = 0

                        for _ in range(repeats):
                            losses, r2, profiling_metrics, _, recon_loss_test, _, _ = run_timevae(
                                X_train, X_test, y_train_scaled, y_test_scaled,
                                timevae_file_path=timevae_file_path,
                                device=device,
                                batch_size=train_cfg["batch_size"],
                                train_epochs=train_cfg["train_epochs"],
                                lr_training=train_cfg["lr"],
                                latent_dim=model_cfg["latent_dim"],
                                hidden_layer_sizes=model_cfg["hidden_layers"],
                                reconstruction_wt=model_cfg["reconstruction_wt"],
                                desired_dataset=cfg.desired_dataset,
                                force_train=True,)

                            rmse_accum     += losses[0]
                            linreg_accum   += losses[1]
                            catboost_accum += losses[2]
                            r2_accum       += r2

                            runtime_accum  += profiling_metrics.get("runtime_s", 0)
                            params_accum   += profiling_metrics.get("num_params_M", 0)
                            flops_accum    += profiling_metrics.get("flops_M", 0)
                            mem_accum      += profiling_metrics.get("peak_memory_MB", 0)

                        losses_avg = [
                            rmse_accum / repeats,
                            linreg_accum / repeats,
                            catboost_accum / repeats]
                        r2_avg = r2_accum / repeats
                        metrics_avg = {
                            "runtime_s": runtime_accum / repeats,
                            "num_params_M": params_accum / repeats,
                            "flops_M": flops_accum / repeats,
                            "peak_memory_MB": mem_accum / repeats,}

                        log_timevae_results(
                            cfg.desired_dataset,
                            window_size,
                            losses_avg,
                            r2_avg,
                            metrics_avg,
                            recon_loss_test,
                            model_cfg,
                            train_cfg,
                            filename=P.timevae_hyperparam_file)

                        print(f"done with config #{counter}")
                        counter += 1


In [ ]:
"ts2vec hyperparam search"

if params["run_console"]["ts2vec"] == True:
    hidden_dims_list = [16] #[8, 12, 16, 20]  # [8, 16, 32]
    latent_dims_list = [12] #[6, 8, 12, 16]  # [8, 16, 32]
    depth_list       = [3] #[3, 4]
    lr_list          = [0.001] #[0.001, 0.05]
    batch_size_list  = [256]

    window_size = 128 #64 
    repeats = 2
    counter = 0

    for hd in hidden_dims_list:
        for ld in latent_dims_list:
            for d in depth_list:
                for lr in lr_list:
                    for bs in batch_size_list:
                        rmse_accum, linreg_accum, catboost_accum, r2_accum = 0, 0, 0, 0
                        runtime_accum, params_accum, flops_accum, mem_accum = 0, 0, 0, 0

                        for _ in range(repeats):
                            model_cfg = {
                                "z_pooling_method": z_pooling_method,
                                "hidden_dims": hd,
                                "latent_dims": ld,
                                "depth": d,}
                            train_cfg = {
                                "lr": lr,
                                "patience": patience,
                                "epochs": epochs,
                                "batch_size": bs,
                                "window_size": window_size,}
                            losses, r2, metrics = run_ts2vec(
                                X_train, X_test, y_train_scaled, y_test_scaled,
                                model_cfg=model_cfg, train_cfg=train_cfg, device=device)
                            rmse_accum    += losses[0]
                            linreg_accum  += losses[1]
                            catboost_accum += losses[2]
                            r2_accum      += r2
                            runtime_accum += metrics['runtime_s']
                            params_accum  += metrics['num_params_M']
                            flops_accum   += metrics['flops_M']
                            mem_accum     += metrics['peak_memory_MB']

                        # average over repeats
                        losses_avg = [rmse_accum / repeats, linreg_accum / repeats, catboost_accum / repeats]
                        r2_avg     = r2_accum / repeats
                        metrics_avg = {
                            'runtime_s': runtime_accum / repeats,
                            'num_params_M': params_accum / repeats,
                            'flops_M': flops_accum / repeats,
                            'peak_memory_MB': mem_accum / repeats}

                        log_ts2vec_results(cfg.desired_dataset, losses_avg, r2_avg, metrics_avg, model_cfg, train_cfg,
                                        filename="results/hyperparam_search_ts2vec.txt")
                        print(f"done with config #{counter}")
                        counter += 1


In [ ]:
"moment param search"

if params["run_console"]["moment"] == True:
    hidden_layers_list = [
        [512, 256, 64, 16],
        [256, 128, 64],
        [1024, 512, 128]]
    unfreeze_last_n_list = [1]
    lr_encoder_list = [0.001]
    lr_head_list    = [0.005, 0.001, 0.05]
    batch_size_list = [128]#, 256]
    dropout_list    = [0.1, 0.3]

    repeats = 2
    counter = 0
    with torch.no_grad():
        moment_model = MOMENTPipeline.from_pretrained(
            f"AutonLab/{params['moment']['model_name']}",
            model_kwargs={'task_name': params['moment']['model_task'], 'n_channels': X_train.shape[2]}).to(device)

    patch_size = getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None)
    X_sample = MomentRunner.pad_to_moment_patch_size(
        torch.tensor(X_train[:1], dtype=torch.float32), patch_size).permute(0, 2, 1).to(device)

    z_sample = moment_model.embed(x_enc=X_sample).embeddings
    latent_dim = z_sample.shape[1]

    # Freeze all encoder parameters
    for p in moment_model.parameters():
        p.requires_grad = False

    # Main hyperparameter search loop
    for hl in hidden_layers_list:
        for unfreeze_n in unfreeze_last_n_list:
            # Unfreeze last N blocks once per config
            MomentRunner._unfreeze_last_n_blocks(moment_model, unfreeze_n)

            for lr_enc in lr_encoder_list:
                for lr_h in lr_head_list:
                    for bs in batch_size_list:
                        for do in dropout_list:

                            losses_accum, r2_accum = 0, 0
                            for _ in range(repeats):
                                rng = np.random.default_rng()
                                rand_seed = rng.integers(0, 2**32 - 1)
                                torch.manual_seed(rand_seed)
                                np.random.seed(rand_seed)
                                random.seed(rand_seed)
                                if device.type.startswith("cuda"):
                                    torch.cuda.manual_seed(rand_seed)
                                    torch.cuda.manual_seed_all(rand_seed)

                                head = make_MLP_regression_head(latent_dim, hl, y_train_scaled, do, device)

                                model_cfg = {
                                    'latent_dim': latent_dim,
                                    'hidden_layers': hl,
                                    'dropout': do,
                                    'unfreeze_last_n': unfreeze_n,
                                    'model_name': params['moment']['model_name'],
                                    'task_name': params['moment']['model_task']}
                                train_cfg = {
                                    'epochs': 10,
                                    'batch_size': bs,
                                    'lr_encoder': lr_enc,
                                    'lr_head': lr_h,
                                    'fine_tune': True}

                                # Train using preloaded encoder and fresh head
                                losses, r2, metrics = MomentRunner.run_moment(
                                    X_train, X_test, y_train_scaled, y_test_scaled,
                                    model_cfg=model_cfg,
                                    train_cfg=train_cfg,
                                    device=device,
                                    seed=rand_seed,
                                    moment_model=moment_model,
                                    head=head)

                                losses_accum += losses[3]  # NN test RMSE
                                r2_accum += r2

                            # Average over repeats
                            losses_avg = [*losses[:3], losses_accum / repeats]
                            r2_avg = r2_accum / repeats

                            MomentRunner.log_moment_results(
                                cfg.desired_dataset,
                                losses_avg,
                                r2_avg,
                                model_cfg=model_cfg,
                                train_cfg=train_cfg,
                                filename="results/hyperparam_search_moment.txt")

                            print(f"done with config #{counter}")
                            counter += 1


In [ ]:
"Test to see why fed-ts2vec does better"

if params["run_console"]["ts2vec_fed"] == True:
    # Single encoder, same total training as federated
    total_epochs = epochs * NUM_FED_SPLITS  # match total updates
    encoder      = TS2VecEncoder(z_pooling=z_pooling_method, lr=ts2vec_lr, device=device, patience=patience)
    encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                depth=ts2vec_depth, batch_size=ts2vec_batch_size,
                n_epochs=total_epochs)
    # Encode
    z_train = encoder.encode(X_train, pooling=None).reshape(X_train.shape[0], -1)
    z_test  = encoder.encode(X_test,  pooling=None).reshape(X_test.shape[0], -1)
    z_train_tensor = torch.tensor(z_train, dtype=torch.float32, device=device)
    z_test_tensor  = torch.tensor(z_test, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
    y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32, device=device)

    # Create regression head
    embedding_dim   = z_train_tensor.shape[1]
    regression_head = make_MLP_regression_head(embedding_dim, cfg.layer1_dim, cfg.layer2_dim, cfg.layer3_dim,
                                        y_train_tensor, cfg.dropout, device)
    # Train + Evaluate
    test_loss = evaluate_MLP_regressor(regression_head, z_train_tensor, z_test_tensor,
                                   y_train_tensor, y_test_tensor, cfg.regressor_epochs, cfg.lr_regressor)
    print(f"Single encoder, extended epochs RMSE: {test_loss:.4f}")
    # ============
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # # Move all latents to same device
    # latents_train = [z.to(device) for z in latents_train]
    # latents_test  = [z.to(device) for z in latents_test]
    # y_train_tensor = y_train_tensor.to(device)
    # y_test_tensor  = y_test_tensor.to(device)

    # # Average latents across encoders
    # Z_train_avg = torch.stack(latents_train, dim=0).mean(dim=0)
    # Z_test_avg  = torch.stack(latents_test,  dim=0).mean(dim=0)

    # # Regression head
    # embedding_dim = Z_train_avg.shape[1]
    # regression_head_avg = make_MLP_regression_head(
    #     embedding_dim, layer1_dim, layer2_dim, layer3_dim, y_train_tensor, dropout, device)

    # # Train + Evaluate
    # test_loss_avg = evaluate_MLP_regressor(
    #     regression_head_avg, Z_train_avg, Z_test_avg,
    #     y_train_tensor, y_test_tensor, regressor_epochs, lr_regressor)
    # print(f"Multiencoder average-latent RMSE: {test_loss_avg:.4f}")


In [ ]:
"Saving to file"
added_entries = []

# if params["run_console"]["direct_preds"]["mean_X"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "mean(X)", mean_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "mean(X)"))
# if params["run_console"]["direct_preds"]["flatten_X"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "flattened(X)", flat_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "flattened(X)"))
# if params["run_console"]["direct_preds"]["custom_row"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "custom(X)", custom_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "custom(X)"))
# if params["run_console"]["direct_preds"]["random_row"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "random(X)", rand_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "random(X)"))

# if params["run_console"]["timevae"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "TimeVAE", timevae_losses, P.json_results_file, result_type="rmse")
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "TimeVAE", [timevae_recon_loss_test], P.json_results_file, result_type="l_recons")
#     # JSONLogger.log_result_to_json(cfg.desired_dataset, "TimeVAE", [timevae_profiling_metrics], P.json_results_file, result_type="profiling")
#     added_entries.append((cfg.desired_dataset, "rmse", "TimeVAE"))
#     added_entries.append((cfg.desired_dataset, "l_recons", "TimeVAE"))
# if params["run_console"]["ts2vec"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "TS2Vec", ts2vec_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "TS2Vec"))
# if params["run_console"]["moment"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "Moment (cent)", moment_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "Moment (cent)"))
# if params["run_console"]["cellsup"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "Cellsup", cellsup_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "Cellsup"))
# if params["run_console"]["barlow_cnn"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "Barlow (CNN)", barlow_cnn_losses, P.json_results_file, result_type="rmse")
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "Barlow (CNN)", [barlow_recon_test], P.json_results_file, result_type="l_recons")
#     added_entries.append((cfg.desired_dataset, "rmse", "Barlow (CNN)"))
# if params["run_console"]["cnn_lstm"]:
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "LSTM (X)", lstm_losses, P.json_results_file, result_type="rmse")
#     JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, "CNN (X)", cnn_mean_losses, P.json_results_file, result_type="rmse")
#     added_entries.append((cfg.desired_dataset, "rmse", "LSTM (X)"))
#     added_entries.append((cfg.desired_dataset, "rmse", "CNN (X)"))

if params["run_console"]["fed"]["ts2vec_fed"]:
    JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, f"TS2Vec ({NUM_FED_SPLITS}-{type_of_split[0].upper()}FL)", ts2vec_fed_losses, P.json_results_file, result_type="rmse")
    added_entries.append((cfg.desired_dataset, "rmse", f"TS2Vec ({NUM_FED_SPLITS}-{type_of_split[0].upper()}FL)"))
if params["run_console"]["fed"]["timevae_fed"]:
    JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, f"TimeVAE ({NUM_FED_SPLITS}-{type_of_split[0].upper()}FL)", timevae_fed_losses, P.json_results_file, result_type="rmse")
    added_entries.append((cfg.desired_dataset, "rmse", f"TimeVAE ({NUM_FED_SPLITS}-{type_of_split[0].upper()}FL)"))
if params["run_console"]["fed"]["moment_fed"]:
    JSONLogger.safe_call(JSONLogger.log_result_to_json, cfg.desired_dataset, f"Moment ({NUM_FED_SPLITS}-{type_of_split[0].upper()}FL)", moment_fed_losses, P.json_results_file, result_type="rmse")
    added_entries.append((cfg.desired_dataset, "rmse", f"Moment ({NUM_FED_SPLITS}-{type_of_split[0].upper()}FL)"))

"fixed latex summary"
data = JSONLogger.load_json_file_safely(P.json_results_file)
# methods_by_type = data.get(cfg.desired_dataset, {})

# for result_type, methods in methods_by_type.items():
#     df = JSONLogger.summarize_runs_to_latex(methods, cfg.num_runs)
#     if df.empty:
#         continue
#     print(f"Added {cfg.desired_dataset} / {result_type} to LaTeX")
#     timestamp = datetime.now().strftime("%H:%M")
#     header = (
#         f"-- {cfg.desired_dataset} {timestamp} "
#         f"{cfg.data_splitting=} {cfg.label_frac=} "
#         f"{cfg.dataset_window=} {result_type=} --\n")
#     latex = df.to_latex(index=False, escape=False)
#     with open(P.latex_results_file, "a") as f:
#         f.write(header)
#         f.write(latex)
#         f.write("\n")
for dataset, result_type, method in added_entries:
    methods = data.get(dataset, {}).get(result_type, {})
    if method not in methods:
        continue
    df = JSONLogger.summarize_runs_to_latex({method: methods[method]}, cfg.num_runs, dataset)
    if df.empty:
        continue
    print(f"Added {dataset} / {result_type} / {method} to LaTeX")
    timestamp = datetime.now().strftime("%H:%M")
    header    = f"-- {dataset} {timestamp} {cfg.data_splitting=} {cfg.label_frac=} {cfg.dataset_window=} {result_type=} --\n"
    latex     = df.to_latex(index=False, escape=False)
    with open(P.latex_results_file, "a") as f:
        f.write(header)
        f.write(latex)
        f.write("\n")

Notifiers.send_discord_message(cfg.webhook_url, "Run finished")
# Notifiers.make_beep_sound(times=3, delay=0.2)


In [ ]:
"BARLOW_AE + BARLOW_AE+SSL"

if params["run_console"]["barlow_ae"] == True:
    latent_dim   = params["barlow"]["ae"]["latent_dim"]
    ssl_weight   = params["barlow"]["ae"]["ssl_weight"]
    augment_const_ae= params["barlow"]["ae"]["augment_const"]

    input_dim = X_train.shape[1] * X_train.shape[2] if X_train.ndim == 3 else X_train.shape[1]
    X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

    # ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim], pred_dim=0).to(device)
    optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

    # ===== TRAIN AE WITH Barlow Twins SSL =====
    start   = time.time()
    ae_model.train()
    for epoch in tqdm(range(train_epochs)):
        optimizer.zero_grad()
        
        X_recon    = ae_model(X_tensor)
        recon_loss = F.mse_loss(X_recon, X_tensor)
        X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
        X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
        v1, v2  = make_two_views_augmentation(X3d, device, augment_const_ae, seed=rand_seed)
        v1_flat = v1.reshape(len(v1), -1)
        v2_flat = v2.reshape(len(v2), -1)
        assert v1_flat.shape[1] == input_dim, f"Expected {input_dim}, got {v1_flat.shape[1]}"

        z1 = ae_model.encode(v1_flat)
        z2 = ae_model.encode(v2_flat)

        # Barlow Twins
        B, D     = z1.shape
        z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
        z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
        xcorr    = (z1_norm.T @ z2_norm) / B
        on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
        off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
        ssl_loss = on_diag + SSL_LAMBDA * off_diag
        loss     = recon_loss + ssl_weight * ssl_loss
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{train_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")

    end   = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    ae_model.eval()
    z_train = get_latent_from_encoder(ae_model, X_L, device=device)
    z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

    # ===== TRAIN CATBOOST & EVALUATE =====
    _, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
    print(f"AE+Barlow latent CatBoost RMSE: {rmse:.4f}")

    _, y_pred_barlow, _, _ = Preds.predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)

    barlow_ae_losses, _ = Preds().evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)
    print(f"Results:\nLinReg: {barlow_ae_losses[0]:.4f} | CatBoost: {barlow_ae_losses[1]:.4f} | RForest: {barlow_ae_losses[2]:.4f}")
    print(f" & \\val{{{barlow_ae_losses[0]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_losses[1]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_losses[2]:.3f}}}{{}} \\\\")


if params["run_console"]["barlow_ae_ssl"] == True:
    # ==== BARLOW TWINS + AE (Semi-Supervised) ====
    latent_dim = 32
    ssl_weight = 1.0
    sup_weight = 0.5  # weight for supervised fine-tuning
    SSL_LAMBDA = 0.005

    # ===== Tensors =====
    # Use only labeled subset
    X_L_tensor = torch.tensor(X_L, dtype=torch.float32, device=device).reshape(len(X_L), -1)
    y_L_tensor = torch.tensor(y_L, dtype=torch.float32, device=device)
    if y_L_tensor.ndim == 1:
        y_L_tensor = y_L_tensor.view(-1, 1)

    # ===== Model =====
    ae_model = ae.FlexibleAutoencoder(
        layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim],pred_dim=1).to(device)
    optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

    # 1️⃣ Stage 1 — Self-Supervised Pretraining (Barlow Twins)
    print("\n=== Stage 1: Self-Supervised Pretraining (Barlow Twins) ===")
    ae_model.train()
    start = time.time()
    for epoch in range(train_epochs):
        optimizer.zero_grad()

        # reconstruction loss
        X_recon = ae_model(X_tensor, mode="reconstruct")
        recon_loss = F.mse_loss(X_recon, X_tensor)

        # augmentations for Barlow Twins
        X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
        X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
        v1, v2  = make_two_views_augmentation(X3d, device, 0.1)
        v1_flat = v1.reshape(len(v1), -1)
        v2_flat = v2.reshape(len(v2), -1)

        # encodings
        z1 = ae_model.encode(v1_flat)
        z2 = ae_model.encode(v2_flat)

        # Barlow Twins loss
        B, D     = z1.shape
        z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
        z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
        xcorr    = (z1_norm.T @ z2_norm) / B
        on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
        off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
        ssl_loss = on_diag + SSL_LAMBDA * off_diag

        # combined SSL + AE loss
        loss = recon_loss + ssl_weight * ssl_loss
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{train_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")
    end = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    # 2️⃣ Stage 2 — Supervised Fine-Tuning (on labeled subset)
    print("\n=== Stage 2: Supervised Fine-Tuning ===")
    for epoch in range(max(5, train_epochs // 2)):
        optimizer.zero_grad()
        y_pred = ae_model(X_L_tensor, mode="predict")  # use forward(mode="predict")
        sup_loss = F.mse_loss(y_pred, y_L_tensor)      # shapes now match
        sup_loss.backward()
        optimizer.step()
        print(f"Fine-tune {epoch+1} - Supervised Loss: {sup_loss.item():.4f}")

    # 3️⃣ Evaluation
    ae_model.eval()
    z_train = get_latent_from_encoder(ae_model, X_L, device=device)
    z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

    barlow_ae_ssl_loss, _ = Preds().evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)

    print(f"\n=== Results ===\nLinReg: {barlow_ae_ssl_loss[0]:.4f} | CatBoost: {barlow_ae_ssl_loss[1]:.4f} | "
        f"Cluster: {barlow_ae_ssl_loss[2]:.4f} | RForest: {barlow_ae_ssl_loss[2]:.4f}")
    print(f" & \\val{{{barlow_ae_ssl_loss[0]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_ssl_loss[1]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_ssl_loss[2]:.3f}}}{{}} \\\\")


In [ ]:
"Load ts2vec latents for clustering"
clustering = False

if clustering == True:
    z_train = np.load(os.path.join(P.ts2vec_params_loc, z_train_file_name))
    z_test  = np.load(os.path.join(P.ts2vec_params_loc, z_test_file_name))
    if cfg.label_frac < 1 and X_U.shape[0] > 0:
        z_U = np.load(os.path.join(P.ts2vec_params_loc, z_U_file_name))

    print(f"z_train: {z_train.shape}, z_test: {z_test.shape}, z_U: {z_U.shape}")
    print(f"y_train: {y_train_scaled.shape}, y_test: {y_test_scaled.shape}")

    # z_train_flat = z_train.reshape(z_train.shape[0], -1)
    # z_test_flat  = z_test.reshape(z_test.shape[0], -1)
    # ========================
    # Flatten over time
    def flat(z: np.ndarray) -> np.ndarray:
        # return z.mean(axis=1)  # (N,D)
        return z.reshape(z.shape[0], -1)

    ZL = flat(z_train)        # labeled latents
    ZU = flat(z_U)            # unlabeled latents
    YL = y_train_scaled.squeeze()

    # 1️⃣ Fit clusters on labeled data only
    k  = int(np.sqrt(len(np.unique(YL))))  # tune as needed
    km = KMeans(n_clusters=k, random_state=0).fit(ZL)

    # 2️⃣ Assign clusters to labeled data
    cL = km.predict(ZL)

    # 3️⃣ Map each cluster to its median y
    cluster2y = {c: np.median(YL[cL==c]) for c in range(k)}

    # 4️⃣ Soft pseudo-labels for unlabeled data
    cU   = km.predict(ZU)
    dist = km.transform(ZU)                        # distance to each cluster
    from scipy.special import softmax
    prob = softmax(-dist / dist.std(), axis=1)     # closer clusters = higher weight
    cluster_values = np.array([cluster2y[c] for c in range(k)])
    YU_soft = prob @ cluster_values                # weighted pseudo-labels

    # Optional: confidence mask
    min_dist = dist.min(axis=1)
    conf_mask = min_dist < np.percentile(min_dist, 50)
    ZU_filtered = ZU[conf_mask]
    YU_filtered = YU_soft[conf_mask]

    # 5️⃣ Augment labeled + pseudo-labeled data
    Z_aug = np.concatenate([ZL, ZU_filtered], axis=0)
    Y_aug = np.concatenate([YL, YU_filtered], axis=0).reshape(-1,1)

    # 6️⃣ Predict on test set
    z_test_concat = flat(z_test)
    _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(
        Z_aug, Y_aug, z_test_concat, y_test_scaled)
    print(f"TS2Vec + KMeans soft pseudo-label RMSE: {rmse:.4f}")


In [ ]:
"params for running Headsup"
# ===== TS2Vec params ======
z_pooling_method   = "mean"
ts2vec_hidden_dims = 16 # units in each layer (> than latent dim)
ts2vec_latent_dims = 8 # latent dim
ts2vec_depth       = 3 # num layers
batch_size  = 32
epochs      = 5 #20
ts2vec_patience    = 25

# predictor_lr           = 0.009
# predictor_epochs       = 50
predictor_dropout      = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

# ===== Headsup pretrain/train params =====
set_all_rand_seeds(42)
batch_size_pretrain  = 16
batch_size_train     = 16

decoder_hidden_dims  = [16, 64, 128]
projection_dim       = 16
lr_pretrain          = 1e-3
lr_train             = 1e-4 #1e-3

patience_pretrain    = 15
patience_train       = 6

train_epochs_pretrain= 8
train_epochs_finetune= 3 # aim for 30-50
warmup_frac_pretrain = 0.05 # 5-10% of pretrain steps
warmup_frac_train    = 0.05 # 5-10% of train steps

weights_pretrain     = {"recon":0.1,"contrast":1.0}
weights_train_100    = {"pred": 1.0, "recon": 0.5, "contrast": 0.5} # 100 is the label_fraction
weights_train_50     = {"pred": 1.0, "recon": 0.1, "contrast": 0.1} # 50 is the label_fraction
weights_train_other  = {"pred": 2.0, "recon": 0.0, "contrast": 0.0}

label_fractions = [1.0, 0.5, 0.25, 0.1]

# === augmentations ===
aug1          = "jitter"
aug1_strength = 0.2
aug2          = "mag_warp"
aug2_strength = 0.1

# === files and names ===
TS2VEC_ENCODER_NAME   = f"ts2vec_encoder_{cfg.desired_dataset}_{ts2vec_hidden_dims}hiddendims_{ts2vec_depth}layers_{ts2vec_latent_dims}dims_{batch_size}batch_{epochs}epoch.pkl"
TS2VEC_ENCODER_FILE   = os.path.join(P.interim_data_loc, "ts2vec_encoders", TS2VEC_ENCODER_NAME)

PRETRAIN_ENCODER_NAME = (f'pretrained_encoder_{cfg.desired_dataset}_lr{lr_pretrain}_epochs{train_epochs_pretrain}_batch{batch_size_pretrain}'
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_pretrain}_frac{"_".join(map(str, label_fractions))}.pth')
PRETRAIN_ENCODER_FILE = os.path.join(P.interim_data_loc, "pretrained_encoders", PRETRAIN_ENCODER_NAME)

EMBEDDING_FILE_NAME   = (f"cached_embeddings_{cfg.desired_dataset}_{cfg.desired_dataset}_lr{lr_train}_epochs{train_epochs_finetune}_batch{batch_size_train}"
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_train}.pth')
EMBEDDING_CACHE_FILE  = os.path.join(P.interim_data_loc, "trained_encoders", EMBEDDING_FILE_NAME)


In [ ]:
"Cellsup: Clustering (L+U), predictor (L), eval (test)"

if params["run_console"]["cellsup"] == True:
    # ===== params + prepare data =====
    num_epochs  = train_epochs #params["cellsup"]["num_epochs"]
    hidden_dim  = X_train.shape[2]//2
    # AE_lr       = params["cellsup"]["AE_lr"]
    # ===== pretrain section =====
    # Pretraining step: each encoder learns X > z > X_recon. After pretraining, encoder is frozen for downstream tasks

    # ===== pretrain AE variants =====
    ae_encoders = {}
    encoders_dims_list = params["cellsup"]["encoders_dims_list"]
    # for i, latent_dim in enumerate(encoders_dims_list):
    #     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    #     ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train, num_epochs=num_epochs, lr=AE_lr,
    #                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
    #     ae_encoders[f"AE_{latent_dim}"] = ae_model

    "even slicing"
    # num_slices = len(encoders_dims_list)
    # for i, latent_dim in enumerate(encoders_dims_list):
    #     X_train_slice   = get_sliced_data(X_train, num_slices, i)
    #     input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
    #     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, 64, latent_dim], pred_dim=0).to(device)
    #     ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train_slice, num_epochs=num_epochs,lr=AE_lr,
    #                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
    #     ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

    "weighted slicing"
    weights  = np.array(encoders_dims_list) / np.sum(encoders_dims_list)
    # X_slices = Slicing.get_weighted_slices(X_train, weights)
    X_slices = Slicing.get_weighted_slices_sqrt(X_train, encoders_dims_list)
    for i, (latent_dim, X_train_slice) in enumerate(zip(encoders_dims_list, X_slices)):
        input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
        ae_model        = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, hidden_dim, latent_dim], pred_dim=0).to(device)
        ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model,X_train_slice,num_epochs=cfg.train_epochs,lr=cfg.AE_lr,
                                                          sample_frac=0.8,weight_decay=cfg.weight_decay,device=device)
        ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

    # ===== pretrain Denoising AE =====
    # denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
    #                                dropout_prob=dropout, noise_std=0.1).to(device)
    # optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr, weight_decay=weight_decay)
    # for epoch in range(num_epochs):
    #     optimizer_dae.zero_grad()
    #     X_recon = denoise_ae(X_tensor)
    #     loss    = F.mse_loss(X_recon, X_tensor)
    #     loss.backward()
    #     optimizer_dae.step()

    # ===== assemble encoders =====
    encoders_dict = {**ae_encoders,
                    #  "denoiseAE": denoise_ae,
                    }

    # Add a suphead for each encoder
    # sup_head_rmse = {}
    # for name, encoder in encoders_dict.items():
    #     rmse = train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test_scaled, dropout,
    #                                       all_encoders=encoders_dict, reg_ortho=1e-3,   # tune this
    #                                       train_encoder=True, device=device, epochs=num_epochs)
    #     sup_head_rmse[name] = rmse
    #     print(f"{name}: RMSE = {rmse:.4f}")

    # ===== Train sup-heads (optionally finetune encoders) =====
    sup_head_rmse = train_sup_heads_joint(encoders_dict, X_L, y_L, X_test, y_test_scaled,
                                          hidden_sizes=[64,32], lr=cfg.AE_lr, epochs=cfg.train_epochs,
                                          device=device, train_encoders=True, reg_ortho=0e-3)
    for name, rmse in sup_head_rmse.items():
        print(f"{name}: RMSE = {rmse:.4f}")

    # xxxxxxxxxx Per-encoder evaluation xxxxxxxxxx
    print("Per-encoder CatBoost RMSE:")
    for name, encoder in encoders_dict.items():
        z_train = Latents.get_latent_tensor(encoder, X_L, train_encoder=False, device=device).cpu().numpy()
        z_test  = Latents.get_latent_tensor(encoder, X_test, train_encoder=False, device=device).cpu().numpy()
        _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
        print(f"   {name}: {rmse:.4f}")

    # ===== Encoder weights =====
    weight_encoding_method = "inverse_rmse"  # "uniform", "inverse_rmse", "softmax"
    encoder_weights = assign_encoder_weights(encoders_dict, sup_head_rmse, weight_encoding_method)

    # ===== ensemble clustering =====
    n_clusters = 8
    ensemble_clusters = Cellsup(encoders_dict=encoders_dict, n_clusters=n_clusters,
                                device=device, cluster_assignment="soft", cluster_metric="ch")

    # """§0 BASELINE: Pure supervised on latents (no clustering, no pseudo-labels)"""
    # z_train_concat = Latents.get_weighted_latents(encoders_dict, X_L, encoder_weights, device=device)
    # z_test_concat  = Latents.get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)
    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_train_concat, y_L, z_test_concat, y_test_scaled)

    # print(f"  >> §0 latent (no cluster z>y) CatBoost RMSE: {rmse:.4f}")
    # linreg_loss0, catboost_loss0, unsupervised_rmse0, rf_rmse0 = \
    #     Preds.evaluate_models_on_dataset(z_train_concat, y_L, z_test_concat, y_test_scaled)

    """§1 Clustering (clusters > pseudo-labels > RMSE)"""
    ensemble_clusters.encoders_dict = encoders_dict
    print("Encoders used for pseudo-labels:", list(ensemble_clusters.encoders_dict.keys()))
    X_all_aug    = np.concatenate([X_L, X_U], axis=0)
    z_all_concat = Latents.get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
    z_test_concat= Latents.get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    ensemble_clusters.fit_kmeans_on_encoder_latents(X_L, encoder_weights=encoder_weights, cluster_range=(cfg.cluster_min, cfg.cluster_max))
    # print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    y_U_pseudo   = ensemble_clusters.assign_pseudo_labels(X_L, y_L, X_U, confidence_thresh=0)
    y_all_aug    = np.concatenate([y_L, y_U_pseudo], axis=0)
    _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    print(f"  >> §1 Semi-supervised latent+cluster CatBoost RMSE: {rmse:.4f}")
    cellsup_losses, _ = Preds().evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    """§2 DeepCluster (z > clusters > rmse)"""
    # ensemble_clusters.cluster_prob_matrix = None
    # multiview_bool = True
    # # if len(encoders_dict) == 1:
    # #     multiview_bool = False

    # if multiview_bool:
    #     X_U_torch    = torch.tensor(X_U, dtype=torch.float32, device=device)
    #     X_U_view1, X_U_view2 = make_two_views_augmentation(X_U_torch, device, scale=0.1)
    #     X_U_aug      = torch.cat([X_U_view1, X_U_view2], dim=0).cpu().numpy()
    #     ensemble_clusters.deepcluster_step_swav(X_U_aug, n_iters=swav_iters, cluster_range=(4, 16),
    #                                             temperature=swav_temp, refine_encoder=False)
    # else:
    #     ensemble_clusters.deepcluster_step_swav(X_U, n_iters=swav_iters, cluster_range=(4,16),
    #                                             temperature=swav_temp, refine_encoder=False)

    # swav_feats_U          = ensemble_clusters.cluster_prob_matrix  # now shape (N_unlabeled, sum_k)
    # encoder_cluster_sizes = [ensemble_clusters.clusterers[name].n_clusters for name in encoders_dict]
    # start = 0
    # per_encoder_means = []
    # for k in encoder_cluster_sizes:
    #     per_encoder_means.append(np.mean(swav_feats_U[:, start:start+k], axis=1, keepdims=True))
    #     start += k
    # y_dim      = y_L.shape[1]
    # y_U_pseudo = np.mean(np.concatenate(per_encoder_means, axis=1), axis=1, keepdims=True)  # (N_unlabeled, 1)
    # y_U_pseudo = y_U_pseudo[:len(X_U)]  

    # y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_dim))  # (N_unlabeled, y_dim)
    # X_all_aug       = np.concatenate([X_L, X_U], axis=0)
    # y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)
    # z_all_concat    = get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
    # z_test_concat   = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    # print(f"  >> §2 DeepCluster latent+cluster CatBoost RMSE: {rmse:.4f}")
    # swav_losses = Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    # """§3 Barlow Twins (SSL consistency regularizer on unlabeled data)"""
    # print(">> Running §3 Barlow Twins consistency step")
    # z_view1_concat = get_weighted_latents(encoders_dict, X_U_view1.cpu().numpy(), encoder_weights, device=device)
    # z_view2_concat = get_weighted_latents(encoders_dict, X_U_view2.cpu().numpy(), encoder_weights, device=device)

    # # compute BT loss (as regularization indicator, not for training)
    # loss_BT = barlow_twins_loss(
    #     torch.tensor(z_view1_concat, device=device, dtype=torch.float32),
    #     torch.tensor(z_view2_concat, device=device, dtype=torch.float32),)
    # print(f"  >> §3 Barlow Twins unsupervised loss: {loss_BT.item():.4f}")

    # # optionally, use BT consistency as pseudo-supervision
    # z_all_concat  = get_weighted_latents(encoders_dict, np.concatenate([X_L, X_U]), encoder_weights, device=device)
    # z_test_concat = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    # # make pseudo-targets = avg 2 BT views’ means (simple consistency trick)
    # y_U_pseudo      = (z_view1_concat.mean(axis=1, keepdims=True) + z_view2_concat.mean(axis=1, keepdims=True))/2
    # y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_L.shape[1]))
    # y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)

    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)
    # print(f"  >> §3 Barlow Twins latent+consistency CatBoost RMSE: {rmse:.4f}")
    # linreg_loss3, catboost_loss3, unsupervised_rmse3, rf_rmse3 = \
    #     Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    print(f"Results for dataset: {cfg.desired_dataset}, {cfg.label_frac=}")
    print("    RMSE       | LinReg | CatBoost | Cluster | RForest")
    # print(f"& Z (concat)   & {linreg_loss0:.4f} & {catboost_loss0:.4f}   & {unsupervised_rmse0:.4f}  & {rf_rmse0:.4f} \\\\")
    print(f"& Z (pseudo)   & {cellsup_losses[0]:.4f} & {cellsup_losses[1]:.4f}   & {cellsup_losses[2]:.4f} \\\\")
    # print(f"& Z (swav)     & {swav_losses[0]:.4f} & {swav_losses[1]:.4f}   & {swav_losses[2]:.4f}  \\\\")
    # print(f"& Z (Barlow)   & {linreg_loss3:.4f} & {catboost_loss3:.4f}   & {unsupervised_rmse3:.4f} & {rf_rmse3:.4f} \\\\")


In [ ]:
"CNN and LSTM solvers"

def X_extract_cnn_features(X, latent_dim=8, channels_1=64, channels_2=64, kernel_size=3, pool_kernel=2,
                         device=device):
    B, T, n_features = X.shape
    X_t = torch.tensor(X, dtype=torch.float32).to(device)

    class CnnEnc(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv1d(n_features, channels_1, kernel_size).to(device)
            self.pool1 = nn.MaxPool1d(pool_kernel)
            self.conv2 = nn.Conv1d(channels_1, channels_2, kernel_size).to(device)
            self.pool2 = nn.MaxPool1d(pool_kernel)
            self.enc_linear = nn.Linear(1, latent_dim)  # placeholder

            # compute flat size with dummy
            with torch.no_grad():
                dummy = torch.zeros(1, n_features, T, device=device)
                x = self.pool1(F.relu(self.conv1(dummy)))
                x = self.pool2(F.relu(self.conv2(x)))
                self.flat_size = x.numel()
                self.flat_shape = x.shape[1:]
            self.enc_linear = nn.Linear(self.flat_size, latent_dim).to(device)

        def encode(self, x):
            x = x.permute(0, 2, 1)
            x = self.pool1(F.relu(self.conv1(x)))
            x = self.pool2(F.relu(self.conv2(x)))
            z = self.enc_linear(x.view(-1, self.flat_size))
            return z

    cnn = CnnEnc().to(device)
    cnn.eval()
    with torch.no_grad():
        features = cnn.encode(X_t).cpu().numpy()
    return cnn, features

def extract_cnn_features(X, latent_dim=8, channels=[64,64], kernel_size=3, pool_kernel=2,
                         device=device):
    _, T, n_features = X.shape
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    cnn = CnnAutoencoder(n_features=n_features, n_timesteps=T, latent_dim=latent_dim,
                         channels=channels, kernel_size=kernel_size, pool_kernel=pool_kernel).to(device)
    cnn.eval()
    with torch.no_grad():
        features = cnn.encode(X_t).cpu().numpy()
    return cnn, features

def train_cnn_head(features_train, y_train, features_test, y_test, lr=1e-3, epochs=100, batch_size=32, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.tensor(features_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(features_test, dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

    head      = nn.Linear(features_train.shape[1], y_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    head.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            y_pred = head(xb)
            loss   = loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()
        if epoch % 10 == 0:
            print(f"CNN Head Epoch {epoch}, Loss: {loss.item():.4f}")

    head.eval()
    with torch.no_grad():
        y_pred_test = head(X_test_t).cpu().numpy()
    rmse = np.sqrt(np.mean((y_test - y_pred_test)**2))
    return head, rmse

def extract_lstm_features(X, hidden_size=64, num_layers=1, device=device):
    B, T, n_features = X.shape
    X_t  = torch.tensor(X, dtype=torch.float32).to(device)
    lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                   batch_first=True).to(device)
    lstm.eval()
    with torch.no_grad():
        _, (hn, _) = lstm(X_t)
        features = hn[-1].cpu().numpy()
    return lstm, features

def train_lstm_head(features_train, y_train, features_test, y_test, lr=1e-3, epochs=50, batch_size=32, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.tensor(features_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(features_test, dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

    head = nn.Linear(features_train.shape[1], y_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    head.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            y_pred = head(xb)
            loss = loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()
        if epoch % 10 == 0:
            print(f"LSTM Head Epoch {epoch}, Loss: {loss.item():.4f}")

    head.eval()
    with torch.no_grad():
        y_pred_test = head(X_test_t).cpu().numpy()
    rmse = np.sqrt(np.mean((y_test - y_pred_test)**2))
    return head, rmse

if params["run_console"]["cnn_lstm"] == True:
    latent_dim     = 32
    latent_dim_lstm= 32
    channels_list = [64, 128]
    kernel_size   = 6#3
    pool_kernel   = 2
    epochs        = 100
    # lr            = 1e-3
    # batch_size = 16

    # ===== CNN =====
    cnn_model, X_train_cnn = extract_cnn_features(X_train, latent_dim=latent_dim, channels=channels_list,
                                                  kernel_size=kernel_size, pool_kernel=pool_kernel, device=device)
    _, X_test_cnn = extract_cnn_features(X_test, latent_dim=latent_dim, channels=channels_list,
                                         kernel_size=kernel_size, pool_kernel=pool_kernel, device=device)
    start = time.time()
    cnn_head, cnn_rmse = train_cnn_head(X_train_cnn, y_L, X_test_cnn, y_test_scaled, epochs=epochs)#, lr=lr)
    print("CNN RMSE:", cnn_rmse)
    cnn_mean_losses, rf_model_cnn = Preds().evaluate_models_on_dataset(X_train_cnn, y_L, X_test_cnn, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    # ===== LSTM =====
    lstm_model, X_train_lstm = extract_lstm_features(X_train, hidden_size=latent_dim_lstm)
    _, X_test_lstm           = extract_lstm_features(X_test, hidden_size=latent_dim_lstm, device=lstm_model.weight_ih_l0.device)
    lstm_head, lstm_rmse     = train_lstm_head(X_train_lstm, y_L, X_test_lstm, y_test_scaled, epochs=epochs)#, lr=lr)

    start = time.time()
    print("LSTM RMSE:", lstm_rmse)
    lstm_losses, rf_model_lstm = Preds().evaluate_models_on_dataset(X_train_lstm, y_L, X_test_lstm, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    print(f"& CNN(X)& {cnn_mean_losses[0]:.4f} & {cnn_mean_losses[1]:.4f} & {cnn_mean_losses[2]:.4f}")
    print(f"& LSTM (X) & {lstm_losses[0]:.4f} & {lstm_losses[1]:.4f}   & {lstm_losses[2]:.4f}")
    print(f"R² (CNN): {rf_model_cnn.score(X_test_cnn, y_test_scaled):.3f}")
    print(f"R² (LSTM): {rf_model_lstm.score(X_test_lstm, y_test_scaled):.3f}")


In [ ]:
"======== code breaker ========"
1 > f


In [ ]:
"autocorr"
from statsmodels.tsa.stattools import acf

X = X_milling
P, R, C = X.shape

LAG_MAX             = int(R / 4)  # Maximum lag: R/4 rule of thumb
CONFIDENCE_INTERVAL = 2 / np.sqrt(R) # Significance threshold: 2/sqrt(R)
DOWNSAMPLE_FACTOR   = 2 # Factor for decimation

for p in range(P):
    for c in range(C):
        # Generate an AR(1) series (time dimension is rows)
        series = np.zeros(R)
        series[0] = np.random.randn()
        for r in range(1, R):
            # Strong positive correlation (phi=0.8)
            series[r] = 0.8 * series[r-1] + np.random.randn() * 0.5
        X[p, :, c] = series

print(f"Time Series Length (R): {R}")
print(f"Maximum Lag (LAG_MAX): {LAG_MAX}")
print(f"Confidence Threshold: +/- {CONFIDENCE_INTERVAL:.3f}\n")

# --- 2. Function to Compute ACF for the 3D Array ---
def compute_3d_acf(data, lag_max):
    """Computes ACF for every (page, col) series."""
    P, _, C = data.shape
    # Initialize the result array: (lag_max + 1, P, C)
    # +1 because lag 0 (autocorr=1) is included
    acf_matrix = np.zeros((lag_max + 1, P, C))

    for p in range(P):
        for c in range(C):
            series = data[p, :, c]            
            acf_values = acf(series, nlags=lag_max, fft=False, adjusted=False)
            acf_matrix[:, p, c] = acf_values
    return acf_matrix

ACF_X = compute_3d_acf(X, LAG_MAX)

# --- 4. Downsample X to get X_prime (Decimation) ---
X_prime = X[:, ::DOWNSAMPLE_FACTOR, :] 

R_prime       = X_prime.shape[1]
LAG_MAX_prime = int(R_prime / 4) # Recalculate based on new R
print(f"Downsampled Length (R'): {R_prime}")
print(f"New Maximum Lag (LAG_MAX'): {LAG_MAX_prime}\n")

# --- 5. Compute ACF for Downsampled Data (X_prime) ---
ACF_X_prime = compute_3d_acf(X_prime, LAG_MAX_prime)

# --- 6. Compact Presentation and Change Assessment ---
# A. Compact Presentation: Average ACF
ACF_AVG_X = np.mean(ACF_X, axis=(1, 2))
ACF_AVG_X_prime = np.mean(ACF_X_prime, axis=(1, 2))

# B. Single-Value Metric: Change in Average Lag-1 Correlation
# Lag 1 is the second element (index 1) in the ACF array
AVG_RHO_1_X       = ACF_AVG_X[1]
AVG_RHO_1_X_prime = ACF_AVG_X_prime[1]
DIFF_RHO_1        = np.abs(AVG_RHO_1_X - AVG_RHO_1_X_prime)

# C. Quantify Change: RMSD of Average ACF Curves (truncated to shorter length)
common_lags = min(len(ACF_AVG_X), len(ACF_AVG_X_prime))
RMSD_ACF    = np.sqrt(np.mean((ACF_AVG_X[:common_lags] - ACF_AVG_X_prime[:common_lags])**2))

print("--- RESULTS ---")
print(f"Original Average Lag-1 Autocorr: {AVG_RHO_1_X:.3f}")
print(f"Downsampled Average Lag-1 Autocorr: {AVG_RHO_1_X_prime:.3f}")
print(f"Absolute Change in Avg Lag-1 Autocorr: {DIFF_RHO_1:.3f}")
print(f"RMSD of Average ACF Curves (up to lag {common_lags-1}): {RMSD_ACF:.3f}")
print("Average ACF (Original vs. Downsampled):")
results_df = pd.DataFrame({
    'Lag': np.arange(common_lags),
    'ACF_X_Avg': ACF_AVG_X[:common_lags],
    'ACF_X_prime_Avg': ACF_AVG_X_prime[:common_lags]}).round(3)
print(results_df.head(10)) # Print first 10 lags for brevity


In [ ]:
"""[almost CORRECT, small issue] Headsup (no predictor q)"""

# once the missing fields are correct, then rely on the moved class in another file, import like so:
# from headsup import Headsup

# ===== init encoder model =====
if os.path.exists(TS2VEC_ENCODER_FILE):
    print("Loading cached TS2Vec encoder...")
    with open(TS2VEC_ENCODER_FILE, "rb") as f:
        ts2vec_encoder = pickle.load(f)
else:
    print("Training TS2Vec encoder...")
    ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device, patience=ts2vec_patience)
    if ts2vec_encoder._stop_early:
        print("Training stopped early due to no improvement.")
    ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                       depth=ts2vec_depth, batch_size=batch_size, n_epochs=epochs)
    with open(TS2VEC_ENCODER_FILE, "wb") as f:
        pickle.dump(ts2vec_encoder, f)

encoder_torch  = TorchWrapper(ts2vec_encoder.ts_model).to(device)
proj_head      = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
decoder        = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                         hidden_sizes=decoder_hidden_dims).to(device) # doesnt belong to encoder
supervised_head= MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
                         hidden_sizes=predictor_hidden_sizes, dropout=predictor_dropout, device=device)
# ===== run Headsup =====
class Headsup:
    def __init__(self, encoder, proj_head, decoder, supervised_head, device, contrast_temp: float = 0.5,
                 aug1: str = "jitter", aug2: str = "mag_warp", aug1_strength: float = 0.1, aug2_strength: float = 0.1,
                 last_block_lr: float = 1e-3, default_lr: float = 1e-4):
        """Wrapper for pretraining + fine-tuning an encoder with projection, decoder, and supervised head
            encoder: nn.Module
            proj_head: nn.Module
            decoder: nn.Module
            supervised_head: wrapper with .model attribute
            device: torch.device
            contrast_temp: float, temperature for NT-Xent loss
            jitter_strength: float, strength of jitter augmentation
            mag_warp_strength: float, strength of mag_warp augmentation
            last_block_lr: float, learning rate for last block when frac < 0.5
            default_lr: float, default learning rate for other params"""
        self.MIN_BATCH_SIZE   = 2 # for contrastive loss equation
        self.PRINT_EVERY      = 3
        self.lr_min           = 1e-5

        self.encoder          = encoder
        self.proj_head        = proj_head
        self.decoder          = decoder
        self.supervised_head  = supervised_head
        self.device           = device

        self.contrast_temp      = contrast_temp
        self.aug1               = aug1
        self.aug2               = aug2
        self.aug1_strength      = aug1_strength
        self.aug2_strength      = aug2_strength
        self.last_block_lr      = last_block_lr
        self.default_lr         = default_lr

    def _augment(self, X):
        X1 = make_augmentations(X, self.aug1, self.device, self.aug1_strength)
        X2 = make_augmentations(X, self.aug2, self.device, self.aug2_strength)
        return X1, X2

    def _early_stop_check(self, loss_total, best_loss, wait, patience):
        """Stop when loss isnt getting better. Returns updated best_loss, wait counter, and a boolean flag indicating whether to stop."""
        if loss_total < best_loss:
            best_loss = loss_total
            wait = 0
            stop = False
        else:
            wait += 1
            stop = wait >= patience
        return best_loss, wait, stop

    def _make_lr_cos_scheduler(self, optimizer, warmup_steps: int, total_steps: int, min_lr: float):
        """Cosine LR scheduler with linear warmup.
            - optimizer: torch optimizer
            - warmup_steps: steps to linearly ramp up LR
            - total_steps: total training steps
            - min_lr: minimum LR at the end of cosine decay"""
        schedulers = []
        for group in optimizer.param_groups:
            base_lr = group["lr"]

            def lr_lambda(step, base_lr=base_lr):
                if step < warmup_steps:
                    return step / float(max(1, warmup_steps))
                progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
                return (min_lr / base_lr) + (1 - min_lr / base_lr) * 0.5 * (1 + math.cos(math.pi * progress))
            schedulers.append(lr_lambda)
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=schedulers)

    def _pretrain_single_epoch(self, X_train, batch_size, weights, optimizer, scheduler):
        """Runs one epoch of pretraining on the encoder."""
        loss_recon, loss_contrast, loss_total = 0, 0, 0
        for i in range(0, len(X_train), batch_size):
            X_batch = torch.tensor(X_train[i:i+batch_size], dtype=torch.float32, device=self.device)
            if X_batch.size(0) < self.MIN_BATCH_SIZE:
                continue

            X1, X2    = self._augment(X_batch)
            z1, z2    = self.encoder(X1), self.encoder(X2)
            h1, h2    = self.proj_head(z1).mean(dim=1), self.proj_head(z2).mean(dim=1)
            z1_pooled = z1.mean(dim=1)
            x_recon   = self.decoder(z1_pooled)

            loss_contrast = norm_temp_xentropy_loss(h1, h2, temperature=self.contrast_temp)
            loss_recon    = F.mse_loss(x_recon, X_batch)
            loss_total    = weights["recon"] * loss_recon + weights["contrast"] * loss_contrast

            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()
            scheduler.step()
        return loss_recon.item(), loss_contrast.item(), loss_total.item()

    def pretrain(self, X_train, batch_size, epochs, warmup_frac_pretrain, weights=weights_pretrain, lr=lr_pretrain, patience = None):
        """Pretrains encoder with contrastive + reconstruction loss. Pretrain usually has lots of steps, and finetuning has few"""
        optimizer_pretrain = torch.optim.AdamW(list(self.encoder.parameters()) +
                                               list(self.proj_head.parameters()) +
                                               list(self.decoder.parameters()), lr=lr)
        max_steps          = train_epochs_pretrain * (len(X_train) // batch_size)
        warmup_steps       = int(warmup_frac_pretrain * max_steps)
        scheduler_pretrain = self._make_lr_cos_scheduler(optimizer_pretrain, warmup_steps=warmup_steps, total_steps=max_steps, min_lr=self.lr_min)

        best_loss, wait = float("inf"), 0
        for epoch in range(epochs):
            loss_recon, loss_contrast, loss_total = self._pretrain_single_epoch(X_train, batch_size, weights, optimizer_pretrain, scheduler_pretrain)
            if epoch % self.PRINT_EVERY == 0:
                print(f"[Pretrain] Epoch {epoch+1}/{epochs}: "
                      f"recon={loss_recon:.4f}, contrast={loss_contrast:.4f}, total={loss_total:.4f}")

            if patience is not None:
                best_loss, wait, stop_flag = self._early_stop_check(loss_total, best_loss, wait, patience)
                if stop_flag:
                    print(f"Early stopping triggered @ epoch {epoch+1}")
                    break
        return self.encoder, self.proj_head, self.decoder

    def _setup_encoder_optimizer(self, frac: float):
        """Sets encoder layers' requires_grad according to labeled fraction.
        Returns weights for the loss components and optimizer"""
        if frac == 1.0:
            for p in self.encoder.parameters():
                p.requires_grad = True # unfrozen encoder
            weights_train   = weights_train_100 #{"pred": 1.0, "recon": 0.5, "contrast": 0.5}
            params_to_opt   = list(self.encoder.parameters()) + list(self.proj_head.parameters()) + \
                              list(self.decoder.parameters()) + list(self.supervised_head.model.parameters())
            optimizer_train = torch.optim.AdamW(params_to_opt, lr=lr_train)
        elif frac >= 0.5:
            for p in self.encoder.parameters():
                p.requires_grad = False # frozen encoder
            weights_train   = weights_train_50 #{"pred": 1.0, "recon": 0.1, "contrast": 0.1}
            params_to_opt   = list(self.encoder.parameters()) + list(self.proj_head.parameters()) + \
                              list(self.decoder.parameters()) + list(self.supervised_head.model.parameters())
            optimizer_train = torch.optim.AdamW(params_to_opt, lr=lr_train)
        else:
            for name, p in self.encoder.named_parameters():
                p.requires_grad = False # frozen encoder
                if name.startswith("encoder_layers") and "last_block" in name:
                    p.requires_grad = True # unfreeze last block only
            weights_train = weights_train_other #{"pred": 2.0, "recon": 0.0, "contrast": 0.0}

            last_block_params = [p for n, p in self.encoder.named_parameters()
                                 if n.startswith("encoder_layers") and "last_block" in n]
            params_to_opt     = list(self.proj_head.parameters()) + list(self.decoder.parameters()) + \
                                list(self.supervised_head.model.parameters())
            optimizer_train   = torch.optim.AdamW([{"params": last_block_params, "lr": self.last_block_lr}, {"params": params_to_opt, "lr": self.default_lr}])
        return weights_train, optimizer_train

    def _train_single_epoch(self, X_L, y_L, X_train, optimizer_train, scheduler_train, batch_size, weights_train):
        """Runs one epoch of fine-tuning on labeled + unlabeled data (tensor conversion done once)."""
        X_L     = X_L.to(self.device) if not isinstance(X_L, torch.Tensor) else X_L
        y_L     = y_L.to(self.device) if not isinstance(y_L, torch.Tensor) else y_L
        X_train = torch.tensor(X_train, dtype=torch.float32, device=self.device) if not isinstance(X_train, torch.Tensor) else X_train

        loss_pred, loss_recon, loss_contrast, loss_total = 0, 0, 0, 0

        num_batches = (len(X_train) + batch_size - 1) // batch_size
        for i in range(num_batches):
            start = i * batch_size
            end_l = min(start + batch_size, len(X_L))
            end_u = min(start + batch_size, len(X_train))

            X_batch_l = X_L[start:end_l]
            y_batch_l = y_L[start:end_l]
            X_batch_u = X_train[start:end_u]

            if X_batch_l.size(0) < self.MIN_BATCH_SIZE or X_batch_u.size(0) < self.MIN_BATCH_SIZE:
                continue

            # encode
            z_l, z_u = self.encoder(X_batch_l), self.encoder(X_batch_u)
            z_l_pooled, z_u_pooled = z_l.mean(dim=1), z_u.mean(dim=1)

            # supervised loss
            y_hat     = self.supervised_head.model(z_l_pooled)
            loss_pred = F.mse_loss(y_hat, y_batch_l)

            # reconstruction loss
            x_recon_l, x_recon_u = self.decoder(z_l_pooled), self.decoder(z_u_pooled)
            loss_recon = (F.mse_loss(x_recon_l, X_batch_l) + F.mse_loss(x_recon_u, X_batch_u)) / 2

            # contrastive loss
            X1_L, X2_L = self._augment(X_batch_l)
            X1_U, X2_U = self._augment(X_batch_u)
            z1_L, z2_L = self.encoder(X1_L), self.encoder(X2_L)
            z1_U, z2_U = self.encoder(X1_U), self.encoder(X2_U)
            h1_L, h2_L = self.proj_head(z1_L).mean(dim=1), self.proj_head(z2_L).mean(dim=1)
            h1_U, h2_U = self.proj_head(z1_U).mean(dim=1), self.proj_head(z2_U).mean(dim=1)
            loss_contrast = (norm_temp_xentropy_loss(h1_L, h2_L, self.contrast_temp) + norm_temp_xentropy_loss(h1_U, h2_U, self.contrast_temp)) / 2

            # backward
            loss_total = weights_train["pred"] * loss_pred + weights_train["recon"] * loss_recon + weights_train["contrast"] * loss_contrast
            optimizer_train.zero_grad()
            loss_total.backward()
            optimizer_train.step()
            scheduler_train.step()
        return loss_pred.item(), loss_recon.item(), loss_contrast.item(), loss_total.item()

    def training_loop(self, X_train, y_train_scaled, X_test, y_test_scaled, batch_size, train_epochs_finetune, warmup_frac_train, label_fractions, patience = None):
        """Fine-tunes encoder + heads over all labeled fractions. Pretrain usually has lots of steps, and finetuning has few"""
        results_dict = {}
        z_train_dict = {}
        z_test_dict  = {}
        y_L_dict     = {}
        for frac in label_fractions:
            n_samples = int(len(X_train) * frac)
            X_L       = torch.tensor(X_train[:n_samples], dtype=torch.float32, device=self.device)
            y_L       = torch.tensor(y_train_scaled[:n_samples], dtype=torch.float32, device=self.device)

            weights_train, optimizer_train = self._setup_encoder_optimizer(frac)
            # scheduler_train = CosineAnnealingLR(optimizer_train, T_max=train_epochs_finetune * (len(X_train)//batch_size), eta_min=self.lr_min)
            max_steps       = train_epochs_finetune * (len(X_train) // batch_size)
            warmup_steps    = int(warmup_frac_train * max_steps)
            scheduler_train = self._make_lr_cos_scheduler(optimizer_train, warmup_steps=warmup_steps, total_steps=max_steps, min_lr=self.lr_min)

            best_loss, wait = float("inf"), 0
            for epoch in range(train_epochs_finetune):
                loss_pred, loss_recon, loss_contrast, loss_total = self._train_single_epoch(
                    X_L, y_L, X_train, optimizer_train, scheduler_train, batch_size, weights_train)
                if epoch % self.PRINT_EVERY == 0:
                    print(f"[Finetune {frac*100:.0f}%] Epoch {epoch+1}/{train_epochs_finetune}: "
                          f"pred={loss_pred:.4f}, recon={loss_recon:.4f}, "
                          f"contrast={loss_contrast:.4f}, total={loss_total:.4f}")
                if patience is not None:
                    best_loss, wait, stop_flag = self._early_stop_check(loss_total, best_loss, wait, patience)
                    if stop_flag:
                        print(f"Early stopping triggered @ epoch {epoch+1} (label frac={frac*100:.0f}%)")
                        break

            #  ====== internal evaluation ======
            self.encoder.eval()
            self.proj_head.eval()
            self.decoder.eval()
            self.supervised_head.model.eval()
            with torch.no_grad():
                z_test             = self.encoder(torch.tensor(X_test, dtype=torch.float32, device=self.device))
                z_test_pooled      = z_test.mean(dim=1)
                y_pred             = self.supervised_head.model(z_test_pooled).cpu().numpy()
                results_dict[frac] = root_mean_squared_error(y_test_scaled, y_pred)

                z_train            = self.encoder(X_L).mean(dim=1).cpu().numpy()
                z_train_dict[frac] = z_train
                z_test_dict[frac]  = z_test_pooled.cpu().numpy()
                y_L_dict[frac]     = y_L.cpu().numpy()
                print(f"Sup. head RMSE ({frac*100:.0f}% labels): {results_dict[frac]:.4f}")
        return results_dict, z_train_dict, z_test_dict, y_L_dict

headsup_model = Headsup(encoder_torch, proj_head, decoder, supervised_head, device,aug1=aug1,
                        aug1_strength=aug1_strength, aug2=aug2, aug2_strength=aug2_strength)
# ===== pretrain ======
if os.path.exists(PRETRAIN_ENCODER_FILE):
    checkpoint = torch.load(PRETRAIN_ENCODER_FILE)
    headsup_model.encoder.load_state_dict(checkpoint["encoder"])
    headsup_model.proj_head.load_state_dict(checkpoint["proj_head"])
    headsup_model.decoder.load_state_dict(checkpoint["decoder"])
    print("Loaded pretrained model.")
else:
    headsup_model.pretrain(X_train, batch_size=batch_size_pretrain, epochs=train_epochs_pretrain,
                           warmup_frac_pretrain=warmup_frac_pretrain, patience=patience_pretrain, weights = weights_pretrain)
    torch.save({"encoder": headsup_model.encoder.state_dict(), "proj_head": headsup_model.proj_head.state_dict(),
                "decoder": headsup_model.decoder.state_dict()}, PRETRAIN_ENCODER_FILE)
    print("Saved pretrained model.")

# ======== train =========
results_dict = {}

if os.path.exists(EMBEDDING_CACHE_FILE):
    with open(EMBEDDING_CACHE_FILE, "rb") as f:
        checkpoint = pickle.load(f)
    results_dict = checkpoint["results_dict"]
    z_train_dict = checkpoint["z_train_dict"]
    z_test_dict  = checkpoint["z_test_dict"]
    y_L_dict     = checkpoint["y_L_dict"]
    print("Loaded finetuned cached embeddings and results.")
else:
    results_dict, z_train_dict, z_test_dict, y_L_dict = headsup_model.training_loop(X_train, y_train_scaled, X_test, y_test_scaled,
                                                                                   batch_size=batch_size_train, patience=patience_train,
                                                                                   train_epochs_finetune=train_epochs_finetune,
                                                                                   warmup_frac_train=warmup_frac_train, label_fractions=label_fractions)
    with open(EMBEDDING_CACHE_FILE, "wb") as f:
        pickle.dump({"results_dict": results_dict, "z_train_dict": z_train_dict, "z_test_dict": z_test_dict, "y_L_dict": y_L_dict}, f)
    print("Saved embeddings and results.")

# ==== downstream / external inference ====
z_train   = z_train_dict[label_frac]
z_test_np = z_test_dict[label_frac]
y_L       = y_L_dict[label_frac]

headsup_loss = Preds.evaluate_models_on_dataset(z_train, y_L, z_test_np, y_test_scaled)

print(f"dataset: {desired_dataset}, method: ts2vec, label_frac: {label_frac}")
print("    RMSE     | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (ts2vec) & {headsup_loss[0]:.4f} & {headsup_loss[1]:.4f}   & {headsup_loss[2]:.4f} \\\\")


In [ ]:
"4 'ablation' scenarios"
# ===== Prepare 2D / embeddings =====
# # Scenario #1 & #4: direct X→y
X_train_mean = X_train.mean(axis=1).astype(np.float32)
X_test_mean  = X_test.mean(axis=1).astype(np.float32)

n_label = int(label_frac * len(X_train_mean))
X_L, y_L = X_train_mean[:n_label], y_train_scaled[:n_label]

# ===== Scenario #1: Direct supervised on 10% labels =====
linreg_1 = LinearRegression().fit(X_L, y_L)
y_pred_1 = linreg_1.predict(X_test_mean)
rmse_1 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_1))

# ===== Scenario #4: Oracle supervised on 100% labels =====
linreg_4 = LinearRegression().fit(X_train_mean, y_train_scaled)
y_pred_4 = linreg_4.predict(X_test_mean)
rmse_4 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_4))

# ===== Scenario #2: Linear probe on pretrained encoder =====
# Freeze encoder, extract embeddings
with torch.no_grad():
    z_train = custom_model.encoder(torch.tensor(X_train[:n_label], dtype=torch.float32, device=device)).mean(dim=1).cpu().numpy()
    z_test  = custom_model.encoder(torch.tensor(X_test, dtype=torch.float32, device=device)).mean(dim=1).cpu().numpy()

# Train simple predictor on embeddings
linreg_2 = LinearRegression().fit(z_train, y_L)
y_pred_2 = linreg_2.predict(z_test)
rmse_2 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_2))

# ===== Scenario #3: Fine-tune pretrained encoder on 10% labels =====
for p in custom_model.encoder.parameters():
    p.requires_grad = True  # unfreeze encoder

results_dict, _, _, _ = custom_model.training_loop(
    X_train, y_train_scaled, X_test, y_test_scaled,
    batch_size=batch_size_train,
    train_epochs_finetune=train_epochs_finetune,
    warmup_frac_train=warmup_frac_train,
    label_fractions=[label_frac])
rmse_3 = results_dict[label_frac]

# ===== Print RMSE table =====
print(f"Scenario | RMSE")
# print(f"#1 Direct X→y (10% labels): {rmse_1:.4f}")
print(f"#2 Linear Probe (encoder frozen): {rmse_2:.4f}")
print(f"#3 Fine-tune (encoder trainable): {rmse_3:.4f}")
# print(f"#4 Oracle X→y (100% labels): {rmse_4:.4f}")


In [ ]:
"SHAP feature importance cell"
from catboost import CatBoostRegressor, Pool
from sklearn.multioutput import MultiOutputRegressor
from sklearn.feature_selection import SelectKBest, mutual_info_regression
import shap

selector = SelectKBest(mutual_info_regression, k=15, random_state=42)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()]

print(f"Kept {len(selected_features)}/48 features:")
print(selected_features.tolist())

# ------------------------------
# 0️⃣ Handle constant targets
constant_targets = np.where(np.std(y_train_scaled, axis=0) == 0)[0]
print("Targets with zero variance:", constant_targets)
y_train_nonconst = np.delete(y_train_scaled, constant_targets, axis=1)
y_test_nonconst  = np.delete(y_test_scaled, constant_targets, axis=1)

# ------------------------------
# 1️⃣ Aggregate timesteps to reduce dimensionality
X_train_flat = X_train.mean(axis=1)  # stations × sensors
X_test_flat  = X_test.mean(axis=1)

# ------------------------------
# 2️⃣ Train initial multi-output model for feature selection
base_model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    task_type="GPU",
    verbose=0,
    random_rand_seed=42
)
multi_model = MultiOutputRegressor(base_model)
multi_model.fit(X_train_flat, y_train_nonconst)

# ------------------------------
# 3️⃣ Feature selection based on importance (average across outputs)
importances_list = [
    estimator.get_feature_importance(Pool(X_train_flat, y_train_nonconst[:, i]))
    for i, estimator in enumerate(multi_model.estimators_)
]
importances = np.mean(importances_list, axis=0)
threshold = np.median(importances)
selected_idx = np.where(importances >= threshold)[0]

X_train_sel = X_train_flat[:, selected_idx]
X_test_sel  = X_test_flat[:, selected_idx]
print("Selected features (sensor indices):", selected_idx)

# ------------------------------
# 4️⃣ Train final multi-output model on selected features
final_base = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    task_type="GPU",
    verbose=0,
    random_rand_seed=42
)
cat_final = MultiOutputRegressor(final_base)
cat_final.fit(X_train_sel, y_train_nonconst)

# ------------------------------
# 5️⃣ Compute SHAP values per target
shap_values_list = []
for estimator in cat_final.estimators_:
    explainer = shap.TreeExplainer(estimator)
    shap_values_list.append(explainer.shap_values(X_test_sel))

# ------------------------------
# 6️⃣ Aggregate SHAP across targets for one summary plot
shap_values_array = np.array(shap_values_list)  # shape: (n_targets, n_samples, n_features)
mean_abs_shap = np.mean(np.abs(shap_values_array), axis=(0,1))  # mean |SHAP| across targets and samples

# ------------------------------
# 7️⃣ Plot aggregated SHAP
plt.figure(figsize=(10,6))
plt.bar([f"f{i}" for i in selected_idx], mean_abs_shap)
plt.xticks(rotation=90)
plt.ylabel("Mean |SHAP value|")
plt.title("Aggregated feature importance across all targets")
plt.show()


In [ ]:
"CELLSUP: NO CLUSTERS"

# ===== params + prepare data =====
label_frac  = 1  # labelled fraction of training data
num_epochs  = 1
AE_lr       = 1e-3
weight_decay= 1e-5
dropout     = 0.1
n_samples   = int(label_frac * len(X_train))
X_L = X_train[:n_samples]  # labelled X
X_U = X_train[n_samples:]  # unlabelled X
y_L = y_train_scaled[:n_samples]  # labelled y

X_all     = np.concatenate([X_L, X_U], axis=0)
input_dim = X_all.shape[1] * X_all.shape[2] if X_all.ndim == 3 else X_all.shape[1]
X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

# ===== pretrain section =====
# Pretraining step: each encoder learns X > z > X_recon. After pretraining, encoder is frozen for downstream tasks

# ===== pretrain AE variants =====
ae_encoders = {}
for latent_dim in [8, 16, 32]:
    ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train, num_epochs=num_epochs, lr=AE_lr, sample_frac=0.8, weight_decay=weight_decay, device=device)
    ae_encoders[f"AE_{latent_dim}"] = ae_model

# --- pretrain Denoising AE ---
denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
                               dropout_prob=0.05, noise_std=0.1).to(device)
optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr, weight_decay=weight_decay)
for epoch in range(num_epochs):
    optimizer_dae.zero_grad()
    X_recon = denoise_ae(X_tensor)
    loss    = F.mse_loss(X_recon, X_tensor)
    loss.backward()
    optimizer_dae.step()

# ===== assemble encoders =====
encoders_dict = {**ae_encoders,
                 "denoiseAE": denoise_ae,
                 }

# oooooooooo Add a suphead for each encoder oooooooooooo
sup_head_rmse = {}

for name, encoder in encoders_dict.items():
    rmse = train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test_scaled, dropout, train_encoder=True, device=device, epochs=num_epochs)
    sup_head_rmse[name] = rmse
    print(f"{name}: RMSE = {rmse:.4f}")
# ooooooooooooooooooooooooooooooooooooooooooooooooooooooo

# rrrrrrrrrrr train/evaluate each encoder individually rrrrrrrrrrr
# For each encoder:
# 1. Encode X_L and X_test to z_train / z_test
# 2. Fit a predictor (CatBoost) from z_train -> y_L
# 3. Predict y_test from z_test using the frozen encoder
# This evaluates the predictive power of each encoder individually
print("Per-encoder CatBoost RMSE:")
for name, encoder in encoders_dict.items():
    z_train = get_latent_tensor(encoder, X_L, train_encoder=False, device=device).cpu().numpy()
    z_test  = get_latent_tensor(encoder, X_test, train_encoder=False, device=device).cpu().numpy()
    _, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
    print(f"   {name}: {rmse:.4f}")

# ===== Encoder weights =====
weight_encoding_method = "inverse_rmse"  # "uniform", "inverse_rmse", "softmax"
if weight_encoding_method == "uniform":
    encoder_weights = {name: 1.0 for name in encoders_dict.keys()}
    total           = sum(encoder_weights.values())
    encoder_weights = {k: v / total for k, v in encoder_weights.items()}
elif weight_encoding_method == "inverse_rmse": # RMSE-based weights: better encoders get higher weight
    encoder_weights = {name: 1/rmse for name, rmse in sup_head_rmse.items()}
    total           = sum(encoder_weights.values())
    encoder_weights = {k: v/total for k,v in encoder_weights.items()}
elif weight_encoding_method == "softmax": # softmax-based weights
    inv_rmse        = np.array([1/r for r in sup_head_rmse.values()])
    weights_softmax = np.exp(inv_rmse) / np.sum(np.exp(inv_rmse))
    encoder_weights = {name: w for name, w in zip(sup_head_rmse.keys(), weights_softmax)}

# ===== ensemble clustering =====
# After pretraining, we encode X_all with all encoders
# Each encoder’s latent z is clustered via KMeans → produces soft assignment vector q_i
# Soft assignments (probabilities) from all encoders are combined (weighted average) 
# → this is the ensemble cluster_prob_matrix (shape: n_samples x n_clusters)
# This cluster_prob_matrix is the central piece for downstream prediction (frozen; no backprop)

# ===== train predictor + evaluate =====
# Use the ensemble cluster probability matrix as features X -> predict y
# Encoders are frozen; predictor (CatBoost or Linear) is trained on top of ensemble features
# Inference for one sample X1:
#   1. Encode X1 via each encoder → z_i
#   2. Map z_i → cluster probabilities q_i
#   3. Fuse q_i across encoders → prob_vector
#   4. Predict y1 = predictor(prob_vector)

# uuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu
print("-------- predict on latents")
def get_concat_latents(encoders_dict, X, device="cpu"):
    """Return concatenated latent vectors from all encoders for X."""
    latents = []
    for name, encoder in encoders_dict.items():
        z = get_latent_tensor(encoder, X, train_encoder=False, device=device).cpu().numpy()
        latents.append(z)
    return np.concatenate(latents, axis=1)  # shape (N, sum(latent_dims))

z_train_concat = get_concat_latents(encoders_dict, X_L, device=device)
z_test_concat  = get_concat_latents(encoders_dict, X_test, device=device)

_, rmse = train_and_eval_catboost(z_train_concat, y_L, z_test_concat, y_test_scaled)
print(f"latent CatBoost RMSE: {rmse:.4f}")

linreg_loss, catboost_loss, rf_rmse = \
    Preds.evaluate_models_on_dataset(z_train_concat, y_train_scaled, z_test_concat, y_test_scaled, label_frac=label_frac)
print(f"Ensemble clustering results: dataset: {desired_dataset}")
print("    RMSE       | LinReg | CatBoost | Cluster | RForest | ElasticNet")
print(f"& Z (concat z) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \\\\")


In [ ]:
"""Headsup for AE"""
from methods.headsup import HeadsupAE

# --- model setup ---
input_dim = X_train.shape[2]
layer_dims = [input_dim, 16]  # encoder dims per timestep
ae_model = ae.FlexibleAutoencoder(layer_dims=layer_dims, pred_dim=y_train_scaled.shape[1],
                                  dropout_prob=0.05, projection_dim=16)
# --- run ---
headsup_model = HeadsupAE(ae_model, device=device, supervised_head=TorchWrapper(ae_model.prediction_head))
headsup_model.pretrain(X_train, batch_size=32, epochs=300)
results_dict, z_train_dict, z_test_dict, y_L_dict = headsup_model.training_loop(X_train, y_train_scaled, X_test, y_test_scaled,
                                                                               batch_size=32, train_epochs_finetune=300,
                                                                               label_fractions=[1.0, 0.5, 0.25, 0.1])
# ==== downstream / external inference ====
frac      = 1.0  # choose fraction to evaluate
z_train   = z_train_dict[frac]
z_test_np = z_test_dict[frac]
y_L       = y_L_dict[frac]

linreg_loss, catboost_loss, unsupervised_rmse, \
    rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(z_train, y_L, z_test_np, y_test_scaled, label_frac=frac)

print(f"dataset: {cfg.desired_dataset}, method: ts2vec")
print("  RMSE   | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (AE) & {linreg_loss:.4f} & {catboost_loss:.4f} & {unsupervised_rmse:.4f} & {rf_rmse:.4f} & {el_rmse:.4f} \\")



In [ ]:
"best results stored here!!!"

"option 1: Loop2 Headsup (no predictor q)"

# ===== params =====
set_rand_seed(42)
train_epochs_pretrain= 150
train_epochs_finetune= 150
batch_size           = 16
decoder_hidden_dims  = [16, 64, 128]
optimizer_lr         = 1e-3
projection_dim       = 16

label_fractions = [1.0, 0.5, 0.25, 0.01]
results_dict    = {}
cb_results_dict = {}

# ===== init models =====
encoder   = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
            depth=ts2vec_depth, batch_size=batch_size, n_epochs=epochs)

proj_head = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
decoder   = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                    hidden_sizes=decoder_hidden_dims).to(device)
sup_head  = MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
                    hidden_sizes=predictor_hidden_sizes, dropout=predictor_dropout, device=device)

# ===== optimizer =====
params    = list(proj_head.parameters()) + list(decoder.parameters()) + list(sup_head.model.parameters())
optimizer = torch.optim.AdamW(params, lr=optimizer_lr)
max_steps = train_epochs_pretrain * (len(X_train) // batch_size)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=1e-5)

# ===== 1) Pretraining loop (contrastive + optional recon) =====
weights = {"pred": 0.0, "recon": 0.1, "contrast": 1.0}
for epoch in range(train_epochs_pretrain):
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]

        # augment
        X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
        X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

        # encode
        z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
        z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

        # projection
        h1, h2 = proj_head(z1), proj_head(z2)

        # losses
        loss_contrast = norm_temp_xentropy_loss(h1, h2, temperature=0.5)
        x_recon       = decoder(z1)
        loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))

        # total weighted loss (no pred loss in pretrain)
        loss = weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    if epoch % 5 == 0:
        print(f"[Pretrain] Epoch {epoch+1}/{train_epochs_pretrain}: recon={loss_recon.item():.4f}, "
              f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# ===== 2) Fine-tuning loop (combined labeled + unlabeled) =====
weights = {"pred": 1.0, "recon": 0.5, "contrast": 0.5}

for frac in label_fractions:
    n_samples = int(len(X_train) * frac)
    X_L = X_train[:n_samples]
    y_L = y_train_scaled[:n_samples]

    for epoch in range(train_epochs_finetune):
        for i in range(0, len(X_train), batch_size):
            # --- Labeled subset in batch ---
            X_batch_l = X_L[i:i+batch_size]
            y_batch_l = y_L[i:i+batch_size]
            if len(X_batch_l) == 0: #skip empty batches
                continue

            # --- Unlabeled subset in batch (fill to batch_size) ---
            start_unlab = i % len(X_train)  # rotate over full dataset
            X_batch_u   = X_train[start_unlab:start_unlab + batch_size]
            if len(X_batch_u) == 0: #skip empty batches
                continue

            # Encode
            z_l = torch.tensor(encoder.encode(X_batch_l), dtype=torch.float32, device=device)
            z_u = torch.tensor(encoder.encode(X_batch_u), dtype=torch.float32, device=device)

            # Supervised loss on labeled
            y_hat     = sup_head.model(z_l)
            loss_pred = F.mse_loss(y_hat, torch.tensor(y_batch_l, dtype=torch.float32, device=device))

            # Reconstruction + contrastive on labeled + unlabeled
            x_recon_l  = decoder(z_l)
            x_recon_u  = decoder(z_u)
            loss_recon = (F.mse_loss(x_recon_l, torch.tensor(X_batch_l, dtype=torch.float32, device=device)) +
                          F.mse_loss(x_recon_u, torch.tensor(X_batch_u, dtype=torch.float32, device=device))) / 2

            # Contrastive
            X1_l = make_augmentations(torch.tensor(X_batch_l, dtype=torch.float32, device=device), "jitter", device, 0.1)
            X2_l = make_augmentations(torch.tensor(X_batch_l, dtype=torch.float32, device=device), "mag_warp", device, 0.1)
            X1_u = make_augmentations(torch.tensor(X_batch_u, dtype=torch.float32, device=device), "jitter", device, 0.1)
            X2_u = make_augmentations(torch.tensor(X_batch_u, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

            z1_l = torch.tensor(encoder.encode(X1_l.cpu().numpy()), dtype=torch.float32, device=device)
            z2_l = torch.tensor(encoder.encode(X2_l.cpu().numpy()), dtype=torch.float32, device=device)
            z1_u = torch.tensor(encoder.encode(X1_u.cpu().numpy()), dtype=torch.float32, device=device)
            z2_u = torch.tensor(encoder.encode(X2_u.cpu().numpy()), dtype=torch.float32, device=device)

            h1_l, h2_l = proj_head(z1_l), proj_head(z2_l)
            h1_u, h2_u = proj_head(z1_u), proj_head(z2_u)
            loss_contrast = (norm_temp_xentropy_loss(h1_l, h2_l, temperature=0.5) +
                             norm_temp_xentropy_loss(h1_u, h2_u, temperature=0.5)) / 2

            # Weighted total
            loss = weights["pred"]*loss_pred + weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

        if epoch % 5 == 0:
            print(f"[Finetune {frac*100:.0f}%] Epoch {epoch+1}/{train_epochs_finetune}: "
                  f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
                  f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

    # ===== inference =====
    proj_head.eval()
    decoder.eval()
    sup_head.model.eval()
    with torch.no_grad():
        z_test = torch.tensor(encoder.encode(X_test), dtype=torch.float32, device=device)
        y_pred = sup_head.model(z_test).cpu().numpy()
        results_dict[frac] = root_mean_squared_error(y_test_scaled, y_pred)
        print(f"RMSE with {frac*100:.0f}% labeled: {results_dict[frac]:.4f}")

        # ==========
        # Encode labeled data for CatBoost training
        z_train = torch.tensor(encoder.encode(X_L), dtype=torch.float32, device=device).cpu().numpy()
        z_test  = z_test.cpu().numpy()  # move to numpy for CatBoost

        # Train & predict with CatBoost
        cb_model, y_pred_cb, rmse_cb, non_constant_idx = Preds().predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
        print(f"RMSE with {frac*100:.0f}% labeled (CatBoost): {rmse_cb:.4f}")
        cb_results_dict[frac] = rmse_cb
        # =======

# --- Display nicely ---
print("\nRMSE for different labeled fractions:")
for frac, rmse_val in results_dict.items():
    print(f"  {int(frac*100):>3}% label: RMSE = {rmse_val:.4f}")

for frac, rmse_val in cb_results_dict.items():
    print(f"  {int(frac*100):>3}% label (CB): RMSE = {rmse_val:.4f}")



# """option 2 (old): Loop2 Headsup (no predictor q)"""

# set_rand_seed(42)

# # ===== params =====
# train_epochs = 50
# batch_size   = 16
# warmup_steps = 1000
# max_steps    = train_epochs * (len(X_train) // batch_size)
# step         = 0

# decoder_hidden_dims = [16, 64, 128]
# optimizer_lr        = 1e-3
# projection_dim      = 16  # smaller than z

# # ===== init models =====
# encoder   = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
# encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
#             depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=epochs)

# proj_head = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
# decoder   = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
#                     hidden_sizes=decoder_hidden_dims).to(device)
# sup_head  = MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
#                     hidden_sizes=predictor_hidden_sizes,
#                     lr=predictor_lr, epochs=1, dropout=predictor_dropout, device=device)

# # ===== optimizer =====
# params    = list(proj_head.parameters()) + list(decoder.parameters()) + list(sup_head.model.parameters())
# optimizer = torch.optim.AdamW(params, lr=optimizer_lr, weight_decay=1e-1) #1e-1 = 0.8586
# # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=1e-5)

# # ===== training loop =====
# for epoch in range(train_epochs):
#     # predictor_lr = schedule_learning_rate(step, max_steps, lr_0=1e-3, lr_end=1e-5, schedule_type="linear")
#     for i in range(0, len(X_train), batch_size):
#         step += 1
#         X_batch, y_batch = X_train[i:i+batch_size], y_train_scaled[i:i+batch_size]

#         # augment
#         X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
#         X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

#         # encode
#         z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
#         z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

#         # projections
#         h1, h2 = proj_head(z1), proj_head(z2)

#         # losses
#         loss_contrast = Losses.compute_byol_loss(h1, h2.detach())  # no predictor q
#         x_recon       = decoder(z1)
#         loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))
#         y_hat         = sup_head.model(z1)
#         loss_pred     = F.mse_loss(y_hat, torch.tensor(y_batch, dtype=torch.float32, device=device))

#         # weighted total loss
#         weights = {"pred": 1.0, "recon": 0.0, "contrast": 0.0}
#         loss = weights["pred"]*loss_pred + weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

#         # backprop
#         optimizer.zero_grad()
#         loss.backward()
#         # torch.nn.utils.clip_grad_norm_(params, 1.0)
#         # torch.nn.utils.clip_grad_norm_(list(sup_head.model.parameters()), max_norm=1.0)
#         optimizer.step()
#         # scheduler.step()

#     print(f"Epoch {epoch+1}/{train_epochs}: "
#           f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
#           f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# # ===== inference =====
# proj_head.eval()
# decoder.eval()
# sup_head.model.eval()

# with torch.no_grad():
#     z_test = torch.tensor(encoder.encode(X_test), dtype=torch.float32, device=device)
#     y_pred = sup_head.model(z_test).cpu().numpy()

# rmse = root_mean_squared_error(y_test_scaled, y_pred)
# print(f"Headsup RMSE: {rmse:.4f}")


In [ ]:
"""[real data] Conditional VAE. Train on train set, inference on test set"""
should_we_include_X = True
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 20
save_path           = f"{P.encoders_folder}/best_cvae.pth"

# === Dataset from df (downsample + split) ===
df_small = df.sample(frac=0.1, random_state=42)  # keep 10%
X = df_small.drop(columns=y_cols + [time_col_name], errors="ignore").values
y = df_small[y_cols].values

# === dont use traintestsplit for timeseries (it randomly shuffles, breaking temporal order)
split_idx       = int(len(df_small) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# === Scaling ===
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train  = x_scaler.fit_transform(X_train)
X_test   = x_scaler.transform(X_test)
y_train  = y_scaler.fit_transform(y_train)
y_test   = y_scaler.transform(y_test)

# === Convert to tensors ===
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)
y_dim   = y_train.shape[1]
x_dim   = X_train.shape[1]

# === Datasets + Loaders ===
train_dataset = TensorDataset(y_train, X_train)
test_dataset  = TensorDataset(y_test, X_test)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, drop_last=True)
val_loader    = DataLoader(test_dataset,  batch_size=batch_size, drop_last=True)

# === Model + Optimizer + Scheduler===
cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=dropout).to(device)
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs,train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Full test set prediction ===
batch_size_pred   = 64  # adjust based on memory
all_y_pred_scaled = []

with torch.no_grad():
    for batch in range(0, len(X_test), batch_size_pred):
        x_batch = X_test[batch : batch + batch_size_pred].to(device)
        if not should_we_include_X:
            x_batch = torch.zeros(x_batch.size(0), x_dim).to(device)
        z_batch             = torch.randn(x_batch.size(0), latent_dim).to(device)
        y_batch_pred_scaled = cond_vae.decode(z_batch, x=x_batch)
        all_y_pred_scaled.append(y_batch_pred_scaled.cpu())

# Concatenate all batches + MSE
y_pred_scaled = torch.cat(all_y_pred_scaled, dim=0).numpy()
mse = mean_squared_error(y_test, y_pred_scaled)
mae = mean_absolute_error(y_test, y_pred_scaled)
print(f"Scaled y: test MSE = {mse:.4f}, test MAE = {mae:.4f}")

y_pred      = y_scaler.inverse_transform(y_pred_scaled)
y_true_orig = y_scaler.inverse_transform(y_test.numpy())

plt.plot(y_true_orig, label="True")
plt.plot(y_pred, label="Predicted")
plt.title(f"Cond. VAE ('{desired_dataset}' dataset)")
plt.xlabel("Timestep")
plt.ylabel("y value")
plt.legend()
plt.show()


# NOTE: consider this repo for Conditional VAE (https://github.com/unnir/cVAE/blob/master/cvae.py)
# or https://freedium.cfd/https://medium.com/@sofeikov/implementing-conditional-variational-auto-encoders-cvae-from-scratch-29fcbb8cb08f

In [ ]:
"""TimesFM"""
from timesfm import TimesFmHparams, TimesFm, TimesFmCheckpoint

# Hyperparameters
hparams = TimesFmHparams(
    backend="jax",
    per_core_batch_size=32,
    horizon_len=128,
    num_layers=20,
    context_len=512,
    use_positional_embedding=True,)

# Local checkpoint folder containing 'checkpoint'
checkpoint = TimesFmCheckpoint(local_dir="interim_data")

# Initialize model
model = TimesFm(hparams=hparams, checkpoint=checkpoint)

# Load manually (if needed)
# model.load_from_checkpoint("interim_data/checkpoint", checkpoint_type=CheckpointType.FLAX)
model.load_from_checkpoint(repo_id="google/timesfm-1.0-200m")#, checkpoint_type=CheckpointType.FLAX)

# Forecast example
y = np.arange(100)
forecast = model.forecast(y, horizon=10)
print(forecast)


In [ ]:
"""Conditional VAE. Train on train set, inference on test set"""

# === Data parameters ===
n_samples  = 1000
y_dim      = 1
x_dim      = 10  # optional
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 10
save_path           = f"{P.encoders_folder}/best_cvae.pth"

# === Dataset ===
y_data  = torch.randn(n_samples, y_dim)
x_data  = torch.randn(n_samples, x_dim)
dataset = TensorDataset(y_data, x_data)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(dataset, batch_size=batch_size)

cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim,
                              latent_dim=latent_dim, dropout=dropout).to(device)

# === Optimizer + Scheduler ===
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs, train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === Load best model for inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Example inference ===
n_rows_gen   = 5
z_sample     = torch.randn(n_rows_gen, latent_dim).to(device)
is_x_present = True
if is_x_present:
    x_sample = torch.randn(n_rows_gen, x_dim).to(device)
else:
    x_sample = torch.zeros(n_rows_gen, x_dim).to(device)

y_sample = cond_vae.decode(z_sample, x=x_sample)
print(f"Generated y sample: {y_sample}")


In [ ]:
"""TimeGPT"""
NIXTLA_API_KEY = 'nixak-TnuMDCHsSM4hajkuXycqZZrNxwtAIoT9O9H7Q8ZwKl2JuJlazRqIPknwJW1AVHX2yB3yfCAwmAogugqQ'
logging.getLogger("nixtla").setLevel(logging.WARNING)

forecaster = TimeGPTForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols, api_key=NIXTLA_API_KEY)

"""single window evaluation"""
print("==== single window evaluation ====")
forecast_dict, _ = forecaster.forecast_timegpt(df_train, df_test, horizon, use_exogenous_cols=True)
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

"""Multi-window evaluation"""
print("==== multi window evaluation ====")
use_exogenous_cols=True
windows_list  = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                           horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_timegpt(train_df, test_df, horizon_len=horizon, use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j] # shape = H
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# Step 3: compute weighted MAE across windows (horizon = weight)
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")

# # Plot:
# client.plot(df, forecast_df, time_col=time_col_name, target_col=y_cols[0], level=[80,90])

# show last 2% of the history + all predictions
# n = int(len(df) * 0.02)
# df_tail = df.tail(n)
# forecast_tail = forecast_df[forecast_df[time_col_name] >= df_tail[time_col_name].iloc[0]]
# client.plot(df_tail,forecast_tail,time_col=time_col_name,target_col=y_cols[0],level=[80, 90])


In [ ]:
"""SARIMAX"""
logging.basicConfig(level=logging.INFO)

forecaster = SARIMAXForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols)

print("==== single window evaluation ====")
forecast_dict, y_true_scaled = forecaster.forecast_sarimax(df_train, df_test, horizon, 
                                                           order=(1, 0, 0), seasonal_order=(0, 0, 0, 0))
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

print("==== multi window evaluation ====")
use_exogenous_cols = False
windows_list = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                          horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_sarimax(train_df, test_df, horizon, order=(1, 0, 0),
                                                               seasonal_order=(0, 0, 0, 0), use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j]
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# ---- Weighted MAE ----
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")
